# SIH NB11: population cascade with CV and temporal TCN

This notebook keeps the proven NB8 full-population model as Stage A. It tests a tiny FIRMS temporal TCN and a fixed conservative rank blend at true population prevalence. Stage B uses the QA-approved Sentinel-2 and WorldCover cohort as a reranking pilot, with handcrafted image features and an optional frozen satellite-pretrained image embedding.

Model selection uses foreign countries only. India is never loaded. The Stage B sample is enriched for EOG matches, so Stage B scores compare branches but do not estimate population precision. The baseline remains active automatically unless a challenger improves macro country PR-AUC across enough countries without a large worst-country loss.

Kaggle settings:

- Accelerator: GPU T4
- Internet: On, only for the optional TorchGeo checkpoint
- Inputs: the saved NB2 output, the latest NB6 v2 output, and the saved NB8 output
- Do not attach NB6 v1 or NB4


In [ ]:
# Optional frozen-image dependency. A failure disables only the SSL challenger.
import subprocess
import sys

commands = [
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "torchgeo==0.7.1"],
    [sys.executable, "-m", "pip", "install", "-q", "timm"],
]
for command in commands:
    completed = subprocess.run(command, check=False)
    if completed.returncode:
        print("Optional SSL dependency setup failed. The core cascade can still run.")
        break

from pathlib import Path
Path("/kaggle/working/nb11_code").mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile /kaggle/working/nb11_code/kg_08_fusion.py
"""Final foreign-country imagery and FIRMS fusion experiment.

All branches use the same QA-approved sources. India is forbidden. The sampled
imagery cohort is deliberately enriched for EOG matches, so its metrics compare
branches but do not estimate population precision.
"""
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import platform
import shutil
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score,
)


PROTOCOL_VERSION = "08-fusion-country-loco-v1"
COUNTRIES = ["Algeria", "Angola", "Indonesia", "Iraq", "Libya", "Nigeria"]
HOLDOUT = "India"
FEATURE_TAG = "2022_2024"

THERMAL_RAW = [
    "active_days_per_year", "active_months_per_year", "det_per_day",
    "det_per_year", "duty_cycle", "span_window_frac", "mean_gap_days",
    "max_gap_days", "modis_per_year", "snpp_per_year", "n_sensors",
    "snpp_modis_ratio", "night_frac", "sat_frac", "lst_mean", "lst_std",
    "frp_cv",
]
THERMAL_RANK = [
    "frp_mean", "frp_max", "frp_med", "frp_p90", "frp_std",
    "frp_dens_mean", "frp_dens_max", "frp_dens_med", "frp_dens_std",
    "t_mir_mean", "t_mir_max", "t_mir_med", "t_mir_std",
    "t_lwir_mean", "t_lwir_max", "t_lwir_med", "t_lwir_std",
    "dt_mir_lwir_mean", "dt_mir_lwir_max", "dt_mir_lwir_med",
    "dt_mir_lwir_std", "frp_sum_per_year",
]
THERMAL_COLS = THERMAL_RAW + [f"{column}_country_pct" for column in THERMAL_RANK]

IMAGE_COLS = (
    [f"img_center_{band}_median" for band in [
        "blue", "green", "red", "nir", "swir16", "swir22"
    ]]
    + [f"img_center_{index}_{stat}" for index in ["ndvi", "ndbi", "mndwi"]
       for stat in ["p10", "median", "p90"]]
    + [f"img_full_{index}_median" for index in ["ndvi", "ndbi", "mndwi"]]
    + [f"img_center_wc_{code}_fraction" for code in [
        10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100
    ]]
    + ["img_nearest_builtup_in_chip_m"]
)
BRANCHES = {
    "thermal_only": THERMAL_COLS,
    "image_only": IMAGE_COLS,
    "early_fusion": THERMAL_COLS + IMAGE_COLS,
}
PARAMS = {
    "objective": "binary", "learning_rate": 0.03, "num_leaves": 7,
    "max_depth": 4, "min_data_in_leaf": 10, "feature_fraction": 0.8,
    "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l1": 2.0,
    "lambda_l2": 10.0, "max_bin": 63, "verbose": -1,
    "num_threads": -1, "force_col_wise": True,
}


def file_hash(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def find_unique(root: Path, name: str) -> Path:
    matches = [path for path in root.rglob(name) if path.is_file()]
    if len(matches) != 1:
        raise FileNotFoundError(f"Need exactly one {name}; found {matches}")
    return matches[0]


def review_mask(series: pd.Series) -> pd.Series:
    """Normalize CSV booleans without treating the string "False" as true."""
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.fillna(False).astype(str).str.strip().str.lower().isin(
        {"true", "1", "yes"}
    )


def load_cohort(input_root: str | Path) -> tuple[pd.DataFrame, dict]:
    input_root = Path(input_root)
    paths = {name: find_unique(input_root, name) for name in [
        "pilot_sources.csv", "image_features.parquet", "image_quality.csv",
        "run_state.json", "feature_manifest.json",
    ]}
    state = json.loads(paths["run_state.json"].read_text(encoding="utf-8"))
    if state.get("holdout_loaded") is not False or state.get("protocol") != "nb6-context-v1":
        raise ValueError("NB6 state is incompatible or loaded the holdout")
    sample = pd.read_csv(paths["pilot_sources.csv"])
    images = pd.read_parquet(paths["image_features.parquet"])
    quality = pd.read_csv(paths["image_quality.csv"])
    if len(sample) != 600 or HOLDOUT in set(sample.country):
        raise ValueError("Expected the frozen 600-source foreign NB6 sample")
    for frame, name in [(sample, "sample"), (images, "images"), (quality, "quality")]:
        if not frame.source_id.is_unique:
            raise ValueError(f"Duplicate source IDs in {name}")
    missing_image = set(IMAGE_COLS) - set(images.columns)
    if missing_image:
        raise ValueError(f"Missing image features: {sorted(missing_image)}")
    cohort = sample.merge(images, on="source_id", how="left", validate="one_to_one")
    cohort = cohort.merge(
        quality[["source_id", "status", "review_reflectance_tail"]],
        on="source_id", how="left", validate="one_to_one",
    )
    review = review_mask(cohort.review_reflectance_tail)
    cohort = cohort.loc[cohort.status.eq("ok") & ~review].copy()
    counts = cohort.groupby("country").agg(
        sources=("source_id", "size"), positives=("is_eog_flare", "sum")
    )
    if set(counts.index) != set(COUNTRIES) or counts.sources.min() < 35 or counts.positives.min() < 5:
        raise ValueError(f"Insufficient country or label coverage:\n{counts}")

    thermal_parts = []
    thermal_hashes = {}
    wanted = ["source_id"] + THERMAL_RAW + THERMAL_RANK
    for country in COUNTRIES:
        path = find_unique(input_root, f"features_{country}_{FEATURE_TAG}.parquet")
        thermal_hashes[path.name] = file_hash(path)
        frame = pd.read_parquet(path, columns=wanted)
        if not frame.source_id.is_unique:
            raise ValueError(f"Duplicate thermal sources for {country}")
        for column in THERMAL_RANK:
            frame[f"{column}_country_pct"] = frame[column].rank(
                method="average", pct=True
            ).astype("float32")
        wanted_ids = set(cohort.loc[cohort.country.eq(country), "source_id"])
        thermal_parts.append(frame.loc[frame.source_id.isin(wanted_ids), ["source_id"] + THERMAL_COLS])
    thermal = pd.concat(thermal_parts, ignore_index=True)
    cohort = cohort.merge(thermal, on="source_id", how="left", validate="one_to_one")
    if cohort[THERMAL_COLS].isna().all(axis=1).any():
        raise ValueError("At least one image source has no matching thermal features")
    for columns in BRANCHES.values():
        values = cohort[columns].to_numpy(dtype="float32")
        if np.isinf(values).any():
            raise ValueError("Infinite model feature found")
    metadata = {
        "nb6_counts": state["counts"],
        "qa_sources": len(cohort),
        "qa_country_label_counts": counts.reset_index().to_dict("records"),
        "input_sha256": {
            **{name: file_hash(path) for name, path in paths.items()},
            **thermal_hashes,
        },
    }
    return cohort.reset_index(drop=True), metadata


def train_model(frame, columns, seed, rounds):
    params = dict(PARAMS)
    params.update({key: seed for key in [
        "seed", "feature_fraction_seed", "bagging_seed", "data_random_seed"
    ]})
    return lgb.train(
        params,
        lgb.Dataset(
            frame[columns].to_numpy(dtype="float32"),
            label=frame.is_eog_flare.to_numpy(dtype="int8"),
            free_raw_data=True,
        ),
        num_boost_round=rounds,
    )


def fit_predict(train, test, columns, seeds, rounds):
    score = np.zeros(len(test), dtype="float64")
    for seed in seeds:
        model = train_model(train, columns, seed, rounds)
        score += model.predict(test[columns].to_numpy(dtype="float32")) / len(seeds)
    return score


def inner_country_oof(train, columns, seed, rounds):
    score = np.empty(len(train), dtype="float64")
    for index, country in enumerate(sorted(train.country.unique())):
        fit = train.loc[train.country.ne(country)]
        test_index = train.index[train.country.eq(country)]
        model = train_model(fit, columns, seed + index * 100, rounds)
        score[test_index] = model.predict(
            train.loc[test_index, columns].to_numpy(dtype="float32")
        )
    return score


def f1_arrays(y, score, thresholds):
    order = np.argsort(score)[::-1]
    y_sorted = y[order]
    score_sorted = score[order]
    tp = np.cumsum(y_sorted)
    fp = np.cumsum(1 - y_sorted)
    total = y.sum()
    positions = np.searchsorted(-score_sorted, -thresholds, side="right") - 1
    valid = positions >= 0
    out = np.zeros(len(thresholds))
    t = np.zeros(len(thresholds)); f = np.zeros(len(thresholds))
    t[valid] = tp[positions[valid]]; f[valid] = fp[positions[valid]]
    out = 2 * t / np.maximum(2 * t + f + total - t, 1)
    return out


def macro_f1_threshold(frame, score):
    thresholds = np.unique(score)
    values = np.zeros(len(thresholds))
    for country in sorted(frame.country.unique()):
        mask = frame.country.eq(country).to_numpy()
        values += f1_arrays(
            frame.loc[mask, "is_eog_flare"].to_numpy(dtype="int8"),
            score[mask], thresholds,
        ) / frame.country.nunique()
    best = int(np.argmax(values))
    return float(thresholds[best]), float(values[best])


def macro_ap(frame, score):
    return float(np.mean([
        average_precision_score(part.is_eog_flare, score[part.index])
        for _, part in frame.groupby("country")
    ]))


def metric_row(frame, score, threshold, branch, held_out):
    y = frame.is_eog_flare.to_numpy(dtype="int8")
    pred = score >= threshold
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "country": held_out, "branch": branch, "n": len(frame),
        "n_positive": int(y.sum()), "threshold": threshold,
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "pr_auc": average_precision_score(y, score),
        "roc_auc": roc_auc_score(y, score),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def run(input_root="/kaggle/input", output_root="/kaggle/working/nb9_final_fusion", rounds=300, seed=131):
    started = time.time()
    output_root = Path(output_root); out = output_root / "outputs"; models = output_root / "models"
    out.mkdir(parents=True, exist_ok=True); models.mkdir(parents=True, exist_ok=True)
    manifest_path = out / "08_manifest.json"
    if manifest_path.exists():
        raise FileExistsError("Use a fresh output directory for a complete fusion run")
    shutil.copy2(Path(__file__), output_root / "kg_08_fusion.py")
    cohort, metadata = load_cohort(input_root)
    rows, prediction_parts, selection_rows = [], [], []
    for outer_index, held_out in enumerate(COUNTRIES):
        train = cohort.loc[cohort.country.ne(held_out)].reset_index(drop=True)
        test = cohort.loc[cohort.country.eq(held_out)].reset_index(drop=True)
        inner_scores = {}
        outer_scores = {}
        for branch, columns in BRANCHES.items():
            inner_scores[branch] = inner_country_oof(
                train, columns, seed + outer_index * 1000, rounds
            )
            threshold, inner_f1 = macro_f1_threshold(train, inner_scores[branch])
            seeds = [seed + 20_000 + outer_index * 1000 + offset for offset in (0, 101, 202)]
            outer_scores[branch] = fit_predict(train, test, columns, seeds, rounds)
            rows.append(metric_row(test, outer_scores[branch], threshold, branch, held_out))
            selection_rows.append({
                "held_out_country": held_out, "branch": branch,
                "inner_macro_ap": macro_ap(train, inner_scores[branch]),
                "inner_macro_f1": inner_f1, "threshold": threshold,
            })
        weights = np.linspace(0, 1, 9)
        weight_ap = [macro_ap(
            train, weight * inner_scores["thermal_only"] + (1 - weight) * inner_scores["image_only"]
        ) for weight in weights]
        weight = float(weights[int(np.argmax(weight_ap))])
        inner_late = weight * inner_scores["thermal_only"] + (1 - weight) * inner_scores["image_only"]
        threshold, inner_f1 = macro_f1_threshold(train, inner_late)
        outer_late = weight * outer_scores["thermal_only"] + (1 - weight) * outer_scores["image_only"]
        rows.append(metric_row(test, outer_late, threshold, "late_fusion", held_out))
        selection_rows.append({
            "held_out_country": held_out, "branch": "late_fusion",
            "inner_macro_ap": max(weight_ap), "inner_macro_f1": inner_f1,
            "threshold": threshold, "thermal_weight": weight,
        })
        part = test[["source_id", "country", "is_eog_flare", "eog_flare_id", "block_id"]].copy()
        for branch, score_values in {**outer_scores, "late_fusion": outer_late}.items():
            part[f"score_{branch}"] = score_values
        prediction_parts.append(part)
        print(f"Completed holdout {held_out}", flush=True)
    metrics = pd.DataFrame(rows)
    predictions = pd.concat(prediction_parts, ignore_index=True)
    selections = pd.DataFrame(selection_rows)
    summary = metrics.groupby("branch").agg(
        macro_f1=("f1", "mean"), macro_pr_auc=("pr_auc", "mean"),
        macro_roc_auc=("roc_auc", "mean"), worst_country_pr_auc=("pr_auc", "min"),
    ).reset_index().sort_values(["macro_pr_auc", "macro_f1"], ascending=False)
    selected = str(summary.iloc[0].branch)
    final_artifacts = []
    if selected == "late_fusion":
        ordered = predictions.set_index("source_id").loc[cohort.source_id]
        weights = np.linspace(0, 1, 9)
        weight_ap = [macro_ap(
            cohort,
            weight * ordered.score_thermal_only.to_numpy()
            + (1 - weight) * ordered.score_image_only.to_numpy(),
        ) for weight in weights]
        final_weight = float(weights[int(np.argmax(weight_ap))])
        final_score = (
            final_weight * ordered.score_thermal_only.to_numpy()
            + (1 - final_weight) * ordered.score_image_only.to_numpy()
        )
        final_branches = ["thermal_only", "image_only"]
    else:
        final_weight = None
        final_score = predictions.set_index("source_id").loc[
            cohort.source_id, f"score_{selected}"
        ].to_numpy()
        final_branches = [selected]
    final_threshold, _ = macro_f1_threshold(cohort, final_score)
    final_seeds = [seed + 30_000 + offset for offset in (0, 101, 202)]
    importance_parts = []
    for branch in final_branches:
        gain = np.zeros(len(BRANCHES[branch]), dtype="float64")
        for index, final_seed in enumerate(final_seeds):
            model = train_model(cohort, BRANCHES[branch], final_seed, rounds)
            path = models / f"{branch}_{index}.txt"; model.save_model(str(path)); final_artifacts.append(path.name)
            gain += model.feature_importance("gain") / len(final_seeds)
        importance_parts.append(pd.DataFrame({
            "branch": branch, "feature": BRANCHES[branch], "mean_gain": gain,
        }))
    metrics.to_csv(out / "08_country_metrics.csv", index=False)
    summary.to_csv(out / "08_branch_summary.csv", index=False)
    selections.to_csv(out / "08_inner_selection.csv", index=False)
    pd.concat(importance_parts, ignore_index=True).sort_values(
        ["branch", "mean_gain"], ascending=[True, False]
    ).to_csv(out / "08_final_feature_importance.csv", index=False)
    predictions.to_parquet(out / "08_loco_predictions.parquet", index=False)
    cohort[["source_id", "country", "is_eog_flare", "eog_flare_id", "block_id"]].to_csv(out / "08_qa_cohort.csv", index=False)
    manifest = {
        "protocol": PROTOCOL_VERSION, "status": "complete", "holdout_country": HOLDOUT,
        "holdout_loaded": False, "interpretation": "enriched foreign imagery pilot; branch comparison, not population precision",
        "branches": {key: value for key, value in BRANCHES.items()}, "params": PARAMS,
        "rounds": rounds, "seed": seed, "outer_ensemble_seeds": 3,
        "selected_branch": selected, "final_threshold": final_threshold,
        "final_late_thermal_weight": final_weight, "model_artifacts": final_artifacts,
        "selection_rule": "highest foreign macro held-out-country AP, then macro F1",
        "metadata": metadata, "elapsed_minutes": (time.time() - started) / 60,
        "python": platform.python_version(), "versions": {
            package: importlib.metadata.version(package)
            for package in ["numpy", "pandas", "lightgbm", "scikit-learn", "pyarrow"]
        },
    }
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return summary, metrics, predictions


if __name__ == "__main__":
    run()


In [ ]:
%%writefile /kaggle/working/nb11_code/kg_10_temporal_tcn.py
"""Foreign-only temporal TCN features for the final SIH fusion model.

The module builds compact 36-month FIRMS sequences, trains a small residual
TCN, and produces country-excluded scores for nested fusion evaluation. India
is rejected at every public boundary. All normalization is fitted on the
training countries for the current exclusion, never on a scored country.
"""
from __future__ import annotations

import gc
import hashlib
import json
import random
from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler


PROTOCOL_VERSION = "10-temporal-tcn-v1"
COUNTRIES = ("Algeria", "Angola", "Indonesia", "Iraq", "Libya", "Nigeria")
HOLDOUT = "India"
WINDOW_YEARS = (2022, 2023, 2024)
FEATURE_TAG = "2022_2024"
MODEL_SENSORS = ("MODIS", "VIIRS_SNPP")
MONTHS = 36
CHANNELS = (
    "log_detection_count",
    "log_active_days",
    "log_frp_sum",
    "log_frp_mean",
    "mir_mean",
    "mir_lwir_mean",
    "night_fraction",
    "saturation_fraction",
    "modis_fraction",
    "presence",
    "month_sin",
    "month_cos",
)
CONTINUOUS_CHANNELS = tuple(range(9))
PRESENCE_CHANNEL = CHANNELS.index("presence")
DEFAULT_ARCHITECTURE = {
    "width": 32,
    "dilations": [1, 2, 4, 8],
    "kernel_size": 3,
    "dropout": 0.15,
}
HARDNESS_COLUMNS = (
    "active_days_per_year",
    "active_months_per_year",
    "det_per_year",
    "night_frac",
    "sat_frac",
    "dt_mir_lwir_mean",
    "frp_mean",
)
FEATURE_COLUMNS = (
    "source_id",
    "country",
    "block_id",
    "is_eog_flare",
    "eog_flare_id",
    *HARDNESS_COLUMNS,
)
DETECTION_COLUMNS = (
    "source_id",
    "acq_dt",
    "frp",
    "t_mir",
    "t_lwir",
    "daynight",
    "sensor",
    "year",
)


def _file_hash(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def _find_equivalent(input_root: Path, name: str) -> tuple[Path, list[str]]:
    """Choose one file, accepting only byte-identical duplicate attachments."""
    matches = sorted(path for path in input_root.rglob(name) if path.is_file())
    if not matches:
        raise FileNotFoundError(f"Missing required input {name}")
    if len(matches) == 1:
        return matches[0], []
    sizes = {path.stat().st_size for path in matches}
    if len(sizes) != 1:
        raise FileNotFoundError(
            f"Conflicting copies of {name}; attach one source: {matches}"
        )
    digests = {_file_hash(path) for path in matches}
    if len(digests) != 1:
        raise FileNotFoundError(
            f"Non-identical copies of {name}; attach one source: {matches}"
        )
    return matches[0], [str(path) for path in matches[1:]]


def _validate_cohort(cohort: pd.DataFrame) -> pd.DataFrame:
    required = {"source_id", "country"}
    missing = required - set(cohort.columns)
    if missing:
        raise ValueError(f"Cohort is missing columns: {sorted(missing)}")
    out = cohort.copy()
    if out.source_id.isna().any() or out.country.isna().any():
        raise ValueError("Cohort source IDs and countries must be non-null")
    out["source_id"] = out.source_id.astype(str)
    out["country"] = out.country.astype(str)
    if not out.source_id.is_unique:
        raise ValueError("Cohort source IDs must be non-null and unique")
    if HOLDOUT in set(out.country) or not set(out.country).issubset(COUNTRIES):
        raise ValueError("Cohort must contain foreign countries only; India is forbidden")
    return out


def _hardness(frame: pd.DataFrame) -> pd.Series:
    ranks = []
    for column in HARDNESS_COLUMNS:
        values = pd.to_numeric(frame[column], errors="coerce")
        values = values.replace([np.inf, -np.inf], np.nan)
        ranks.append(values.rank(method="average", pct=True, na_option="bottom"))
    return pd.concat(ranks, axis=1).mean(axis=1)


def _sample_country(
    frame: pd.DataFrame,
    negative_ratio: int,
    seed: int,
) -> tuple[pd.DataFrame, dict]:
    """Select one source per known site and a hard/random unlabeled mixture."""
    country = str(frame.country.iloc[0])
    if not frame.country.eq(country).all() or country not in COUNTRIES:
        raise ValueError(f"Invalid country frame: {country}")
    if (
        not frame.source_id.is_unique
        or frame.block_id.isna().any()
        or not frame.is_eog_flare.isin([0, 1]).all()
    ):
        raise ValueError(f"Invalid identifiers or labels for {country}")

    known_positive = frame.loc[frame.is_eog_flare.eq(1)].copy()
    if known_positive.empty or known_positive.eog_flare_id.isna().any():
        raise ValueError(f"Known positives and site identifiers are required for {country}")
    positive_blocks = set(known_positive.block_id)
    positive = known_positive.sort_values(
        ["active_days_per_year", "det_per_year", "source_id"],
        ascending=[False, False, True],
        kind="stable",
    ).drop_duplicates("eog_flare_id", keep="first")
    positive["sample_role"] = "positive_site"
    positive["hardness"] = np.nan

    unlabeled_all = frame.loc[frame.is_eog_flare.eq(0)].copy()
    positive_block_overlap = unlabeled_all.block_id.isin(positive_blocks)
    excluded_positive_block_rows = int(positive_block_overlap.sum())
    unlabeled = unlabeled_all.loc[~positive_block_overlap].copy()
    unlabeled["hardness"] = _hardness(unlabeled)
    target = min(len(unlabeled), negative_ratio * len(positive))
    if target < 1:
        raise ValueError(f"No unlabeled training examples available for {country}")
    hard_count = (target + 1) // 2
    hard = unlabeled.sort_values(
        ["hardness", "source_id"], ascending=[False, True], kind="stable"
    ).head(hard_count).copy()
    hard["sample_role"] = "hard_unlabeled"

    remainder = unlabeled.loc[~unlabeled.source_id.isin(hard.source_id)].copy()
    random_count = target - len(hard)
    if random_count:
        rng = np.random.default_rng(seed)
        positions = rng.choice(len(remainder), size=random_count, replace=False)
        sampled_random = remainder.iloc[np.sort(positions)].copy()
    else:
        sampled_random = remainder.head(0).copy()
    sampled_random["sample_role"] = "random_unlabeled"

    selected = pd.concat([positive, hard, sampled_random], ignore_index=True)
    if not selected.source_id.is_unique:
        raise ValueError(f"Sampling produced duplicate sources for {country}")
    diagnostic = {
        "country": country,
        "known_positive_sources": len(known_positive),
        "unique_positive_sites": len(positive),
        "available_unlabeled_before_block_guard": len(unlabeled_all),
        "positive_block_unlabeled_excluded": excluded_positive_block_rows,
        "available_unlabeled_after_block_guard": len(unlabeled),
        "selected_hard_unlabeled": len(hard),
        "selected_random_unlabeled": len(sampled_random),
    }
    return selected, diagnostic


def _monthly_sequence(detections: pd.DataFrame, source_ids: Sequence[str]) -> np.ndarray:
    source_ids = [str(value) for value in source_ids]
    positions = {source_id: index for index, source_id in enumerate(source_ids)}
    sequence = np.zeros((len(source_ids), len(CHANNELS), MONTHS), dtype="float32")

    month_number = np.arange(MONTHS) % 12
    angle = 2.0 * np.pi * month_number / 12.0
    sequence[:, CHANNELS.index("month_sin"), :] = np.sin(angle).astype("float32")
    sequence[:, CHANNELS.index("month_cos"), :] = np.cos(angle).astype("float32")
    if detections.empty:
        return sequence

    frame = detections.copy()
    frame["source_id"] = frame.source_id.astype(str)
    frame = frame.loc[frame.source_id.isin(positions)].copy()
    if frame.empty:
        return sequence
    frame["acq_dt"] = pd.to_datetime(frame.acq_dt, errors="coerce")
    frame = frame.loc[
        frame.acq_dt.dt.year.isin(WINDOW_YEARS)
        & frame.sensor.astype(str).isin(MODEL_SENSORS)
    ].copy()
    if frame.empty:
        return sequence
    frame["month_index"] = (
        (frame.acq_dt.dt.year - WINDOW_YEARS[0]) * 12
        + frame.acq_dt.dt.month - 1
    ).astype("int16")
    frame["day"] = frame.acq_dt.dt.normalize()
    frame["mir_lwir"] = frame.t_mir - frame.t_lwir
    frame["is_night"] = frame.daynight.astype(str).str.upper().eq("N").astype("float32")
    frame["is_saturated"] = frame.t_mir.ge(367.0).astype("float32")
    frame["is_modis"] = frame.sensor.astype(str).eq("MODIS").astype("float32")

    grouped = frame.groupby(["source_id", "month_index"], observed=True)
    monthly = grouped.agg(
        detection_count=("frp", "size"),
        active_days=("day", "nunique"),
        frp_sum=("frp", "sum"),
        frp_mean=("frp", "mean"),
        mir_mean=("t_mir", "mean"),
        mir_lwir_mean=("mir_lwir", "mean"),
        night_fraction=("is_night", "mean"),
        saturation_fraction=("is_saturated", "mean"),
        modis_fraction=("is_modis", "mean"),
    ).reset_index()
    monthly = monthly.replace([np.inf, -np.inf], np.nan)

    row = monthly.source_id.map(positions).to_numpy(dtype="int64")
    month = monthly.month_index.to_numpy(dtype="int64")
    if not ((month >= 0) & (month < MONTHS)).all():
        raise ValueError("Detection outside the fixed 2022 to 2024 window")
    log_columns = (
        "detection_count", "active_days", "frp_sum", "frp_mean"
    )
    for channel, column in enumerate(log_columns):
        values = pd.to_numeric(monthly[column], errors="coerce").fillna(0)
        values = np.log1p(np.maximum(values.to_numpy(dtype="float32"), 0))
        sequence[row, channel, month] = values
    for channel, column in enumerate((
        "mir_mean", "mir_lwir_mean", "night_fraction",
        "saturation_fraction", "modis_fraction",
    ), start=4):
        values = pd.to_numeric(monthly[column], errors="coerce").fillna(0)
        sequence[row, channel, month] = values.to_numpy(dtype="float32")
    sequence[row, PRESENCE_CHANNEL, month] = 1.0
    return sequence


def _descriptors(meta: pd.DataFrame, sequences: np.ndarray) -> pd.DataFrame:
    present = sequences[:, PRESENCE_CHANNEL, :] > 0.5
    counts = np.expm1(sequences[:, CHANNELS.index("log_detection_count"), :])
    active_days = np.expm1(sequences[:, CHANNELS.index("log_active_days"), :])
    frp_sum = np.expm1(sequences[:, CHANNELS.index("log_frp_sum"), :])
    night = sequences[:, CHANNELS.index("night_fraction"), :]
    saturation = sequences[:, CHANNELS.index("saturation_fraction"), :]
    modis = sequences[:, CHANNELS.index("modis_fraction"), :]
    denominator = np.maximum(counts.sum(axis=1), 1.0)
    adjacent = (present[:, 1:] & present[:, :-1]).sum(axis=1)
    possible = np.maximum(present.sum(axis=1) - 1, 1)
    descriptor = meta[[
        "source_id", "country", "is_eog_flare", "sample_role", "train_selected",
        "is_cohort",
    ]].copy()
    descriptor["ts_detection_count"] = counts.sum(axis=1).astype("float32")
    descriptor["ts_active_days"] = active_days.sum(axis=1).astype("float32")
    descriptor["ts_active_months"] = present.sum(axis=1).astype("int16")
    descriptor["ts_frp_sum"] = frp_sum.sum(axis=1).astype("float32")
    descriptor["ts_mir_mean"] = (
        (sequences[:, CHANNELS.index("mir_mean"), :] * counts).sum(axis=1)
        / denominator
    ).astype("float32")
    descriptor["ts_night_fraction"] = (
        (night * counts).sum(axis=1) / denominator
    ).astype("float32")
    descriptor["ts_saturation_fraction"] = (
        (saturation * counts).sum(axis=1) / denominator
    ).astype("float32")
    descriptor["ts_modis_fraction"] = (
        (modis * counts).sum(axis=1) / denominator
    ).astype("float32")
    descriptor["ts_month_continuity"] = (adjacent / possible).astype("float32")
    return descriptor


def prepare_temporal_data(
    input_root: str | Path,
    cohort: pd.DataFrame,
    negative_ratio: int = 10,
    seed: int = 4103,
    include_population: bool = True,
) -> tuple[pd.DataFrame, np.ndarray, pd.DataFrame, dict]:
    """Build sampled foreign training data and aligned 36-month sequences.

    The returned ``meta`` and ``sequences`` share row order. ``meta`` contains
    every sampled training source plus every requested cohort source. By
    default it also contains the complete foreign common-window population so
    single-country exclusions can be measured at real prevalence.
    """
    input_root = Path(input_root)
    cohort = _validate_cohort(cohort)
    if not isinstance(negative_ratio, int) or negative_ratio < 1:
        raise ValueError("negative_ratio must be a positive integer")

    selected_parts = []
    sampling_diagnostics = []
    cohort_feature_parts = []
    population_feature_parts = []
    selected_paths: dict[str, str] = {}
    duplicate_paths: dict[str, list[str]] = {}
    input_hashes: dict[str, str] = {}
    for country_index, country in enumerate(COUNTRIES):
        name = f"features_{country}_{FEATURE_TAG}.parquet"
        path, duplicates = _find_equivalent(input_root, name)
        frame = pd.read_parquet(path, columns=list(FEATURE_COLUMNS))
        if frame.source_id.isna().any() or frame.country.isna().any():
            raise ValueError(f"Null source ID or country in {path}")
        frame["source_id"] = frame.source_id.astype(str)
        frame["country"] = frame.country.astype(str)
        if not frame.country.eq(country).all():
            raise ValueError(f"Wrong country content in {path}")
        selected_country, sampling_diagnostic = _sample_country(
            frame, negative_ratio, seed + country_index * 1009
        )
        selected_parts.append(selected_country)
        sampling_diagnostics.append(sampling_diagnostic)
        wanted_ids = set(cohort.loc[cohort.country.eq(country), "source_id"])
        cohort_feature_parts.append(frame.loc[
            frame.source_id.isin(wanted_ids),
            ["source_id", "country", "block_id", "is_eog_flare", "eog_flare_id"],
        ].copy())
        if include_population:
            population_feature_parts.append(frame[[
                "source_id", "country", "block_id", "is_eog_flare", "eog_flare_id"
            ]].copy())
        selected_paths[name] = str(path)
        duplicate_paths[name] = duplicates
        input_hashes[name] = _file_hash(path)
        del frame
        gc.collect()

    selected = pd.concat(selected_parts, ignore_index=True)
    if not selected.source_id.is_unique:
        raise ValueError("Sampled source IDs must be globally unique")
    selected["train_selected"] = True

    cohort_features = pd.concat(cohort_feature_parts, ignore_index=True)
    cohort_lookup = cohort[["source_id", "country"]].merge(
        cohort_features,
        on=["source_id", "country"], how="left", validate="one_to_one",
    )
    if cohort_lookup.is_eog_flare.isna().any():
        missing = cohort_lookup.loc[cohort_lookup.is_eog_flare.isna(), "source_id"]
        raise ValueError(f"Cohort source missing from common-window features: {missing.tolist()}")
    if "is_eog_flare" in cohort:
        supplied = cohort.set_index("source_id").is_eog_flare
        expected = cohort_lookup.set_index("source_id").is_eog_flare
        if not supplied.astype("int8").equals(expected.astype("int8")):
            raise ValueError("Cohort labels disagree with common-window features")

    meta_columns = [
        "source_id", "country", "block_id", "is_eog_flare", "eog_flare_id",
        "sample_role", "train_selected",
    ]
    selected_meta = selected[meta_columns].copy()
    if include_population:
        base = pd.concat(population_feature_parts, ignore_index=True)
        if not base.source_id.is_unique:
            raise ValueError("Population source IDs must be globally unique")
        extra_role = "population_only"
    else:
        base = cohort_lookup.copy()
        extra_role = "cohort_only"
    extra = base.loc[~base.source_id.isin(selected_meta.source_id)].copy()
    extra["sample_role"] = extra_role
    extra["train_selected"] = False
    meta = pd.concat([selected_meta, extra[meta_columns]], ignore_index=True)
    meta = meta.sort_values(["country", "source_id"], kind="stable").reset_index(drop=True)
    meta["is_eog_flare"] = meta.is_eog_flare.astype("int8")
    meta["is_cohort"] = meta.source_id.isin(cohort.source_id)
    meta["population_complete"] = bool(include_population)
    if not meta.source_id.is_unique or HOLDOUT in set(meta.country):
        raise ValueError("Aligned temporal metadata is invalid")
    del (
        selected_parts, cohort_feature_parts, population_feature_parts,
        cohort_features, selected, cohort_lookup, base, extra,
    )
    gc.collect()

    sequences = np.zeros((len(meta), len(CHANNELS), MONTHS), dtype="float32")
    month_number = np.arange(MONTHS) % 12
    sequences[:, CHANNELS.index("month_sin"), :] = np.sin(
        2.0 * np.pi * month_number / 12.0
    ).astype("float32")
    sequences[:, CHANNELS.index("month_cos"), :] = np.cos(
        2.0 * np.pi * month_number / 12.0
    ).astype("float32")

    for country in COUNTRIES:
        name = f"detections_{country}.parquet"
        path, duplicates = _find_equivalent(input_root, name)
        country_ids = meta.loc[meta.country.eq(country), "source_id"].tolist()
        detections = pd.read_parquet(
            path,
            columns=list(DETECTION_COLUMNS),
            filters=[("year", ">=", WINDOW_YEARS[0]), ("year", "<=", WINDOW_YEARS[-1])],
        )
        country_sequence = _monthly_sequence(detections, country_ids)
        indices = meta.index[meta.country.eq(country)].to_numpy(dtype="int64")
        sequences[indices] = country_sequence
        del detections, country_sequence
        gc.collect()
        selected_paths[name] = str(path)
        duplicate_paths[name] = duplicates
        input_hashes[name] = _file_hash(path)

    if not np.isfinite(sequences).all():
        raise ValueError("Temporal sequences contain non-finite values")
    if (sequences[:, PRESENCE_CHANNEL, :].sum(axis=1) == 0).any():
        missing = meta.loc[
            sequences[:, PRESENCE_CHANNEL, :].sum(axis=1) == 0, "source_id"
        ]
        raise ValueError(f"Sources without common-window detections: {missing.tolist()}")
    descriptors = _descriptors(meta, sequences)

    sample_counts = (
        meta.loc[meta.train_selected]
        .groupby(["country", "sample_role"], observed=True)
        .size().rename("sources").reset_index().to_dict("records")
    )
    cohort_signature = hashlib.sha256(
        cohort[["source_id", "country"]]
        .sort_values(["country", "source_id"])
        .to_csv(index=False).encode("utf-8")
    ).hexdigest()
    metadata = {
        "protocol": PROTOCOL_VERSION,
        "countries": list(COUNTRIES),
        "holdout": HOLDOUT,
        "holdout_loaded": False,
        "window_years": list(WINDOW_YEARS),
        "channels": list(CHANNELS),
        "sequence_shape": [len(CHANNELS), MONTHS],
        "negative_ratio": negative_ratio,
        "hard_fraction": 0.5,
        "seed": seed,
        "sample_counts": sample_counts,
        "sampling_diagnostics": sampling_diagnostics,
        "training_sources": int(meta.train_selected.sum()),
        "cohort_sources": len(cohort),
        "population_included": bool(include_population),
        "aligned_sources": len(meta),
        "cohort_sha256": cohort_signature,
        "input_paths": selected_paths,
        "ignored_identical_duplicates": duplicate_paths,
        "input_sha256": input_hashes,
    }
    return meta, sequences, descriptors, metadata


def _seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


def _resolve_device(device: str | torch.device | None) -> torch.device:
    if device is None:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    resolved = torch.device(device)
    if resolved.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable")
    return resolved


def _bag_training_mask(
    meta: pd.DataFrame,
    excluded_countries: Sequence[str],
    unlabeled_keep_fraction: float,
    seed: int,
) -> np.ndarray:
    """Keep all positive sites and a deterministic fraction of each PU pool."""
    if not 0 < unlabeled_keep_fraction <= 1:
        raise ValueError("unlabeled_keep_fraction must be in (0, 1]")
    eligible = (
        meta.train_selected.to_numpy(dtype=bool)
        & ~meta.country.isin(excluded_countries).to_numpy()
    )
    labels = meta.is_eog_flare.to_numpy(dtype="int8")
    keep = eligible & labels.astype(bool)
    rng = np.random.default_rng(seed)
    unlabeled_positions = np.flatnonzero(eligible & (labels == 0))
    unlabeled = meta.iloc[unlabeled_positions][["country", "sample_role"]].copy()
    unlabeled["row_position"] = unlabeled_positions
    expected_roles = {"hard_unlabeled", "random_unlabeled"}
    if not set(unlabeled.sample_role).issubset(expected_roles):
        raise ValueError("Unexpected unlabeled sample role in TCN training pool")
    for _, part in unlabeled.groupby(["country", "sample_role"], observed=True):
        positions = part.row_position.to_numpy(dtype="int64")
        count = max(1, int(round(unlabeled_keep_fraction * len(positions))))
        chosen = rng.choice(positions, size=min(count, len(positions)), replace=False)
        keep[chosen] = True
    return keep


@dataclass(frozen=True)
class Normalization:
    center: np.ndarray
    scale: np.ndarray

    def transform(self, sequences: np.ndarray) -> np.ndarray:
        values = np.asarray(sequences, dtype="float32").copy()
        if values.ndim != 3 or values.shape[1:] != (len(CHANNELS), MONTHS):
            raise ValueError(f"Expected [N,{len(CHANNELS)},{MONTHS}] sequences")
        observed = values[:, PRESENCE_CHANNEL, :] > 0.5
        for channel in CONTINUOUS_CHANNELS:
            normalized = (values[:, channel, :] - self.center[channel]) / self.scale[channel]
            values[:, channel, :] = np.where(observed, normalized, 0.0)
            np.clip(
                values[:, channel, :], -8.0, 8.0,
                out=values[:, channel, :],
            )
        return values

    def to_dict(self) -> dict:
        return {
            "center": self.center.astype(float).tolist(),
            "scale": self.scale.astype(float).tolist(),
        }


def _fit_normalization(sequences: np.ndarray) -> Normalization:
    values = np.asarray(sequences, dtype="float32")
    observed = values[:, PRESENCE_CHANNEL, :] > 0.5
    if not observed.any():
        raise ValueError("Training sequences have no observed months")
    center = np.zeros(len(CHANNELS), dtype="float32")
    scale = np.ones(len(CHANNELS), dtype="float32")
    for channel in CONTINUOUS_CHANNELS:
        channel_values = values[:, channel, :][observed]
        channel_values = channel_values[np.isfinite(channel_values)]
        if not len(channel_values):
            raise ValueError(f"No finite values for channel {CHANNELS[channel]}")
        center[channel] = np.median(channel_values)
        q25, q75 = np.quantile(channel_values, [0.25, 0.75])
        width = float(q75 - q25)
        if width < 1e-4:
            width = float(np.std(channel_values))
        scale[channel] = width if width >= 1e-4 else 1.0
    return Normalization(center=center, scale=scale)


class ResidualTCNBlock(nn.Module):
    def __init__(self, width: int, dilation: int, dropout: float) -> None:
        super().__init__()
        self.convolution1 = nn.Conv1d(
            width, width, kernel_size=3, padding=dilation, dilation=dilation
        )
        self.normalization1 = nn.GroupNorm(4, width)
        self.convolution2 = nn.Conv1d(
            width, width, kernel_size=3, padding=dilation, dilation=dilation
        )
        self.normalization2 = nn.GroupNorm(4, width)
        self.activation = nn.SiLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        residual = values
        values = self.convolution1(values)
        values = self.normalization1(values)
        values = self.dropout(self.activation(values))
        values = self.convolution2(values)
        values = self.normalization2(values)
        return self.activation(residual + self.dropout(values))


class TinyTemporalTCN(nn.Module):
    """Small sequence classifier with a 61-month nominal receptive field."""

    def __init__(
        self,
        input_channels: int = len(CHANNELS),
        width: int = 32,
        dilations: Sequence[int] = (1, 2, 4, 8),
        dropout: float = 0.15,
    ) -> None:
        super().__init__()
        if width % 4:
            raise ValueError("TCN width must be divisible by four")
        self.projection = nn.Conv1d(input_channels, width, kernel_size=1)
        self.blocks = nn.Sequential(*[
            ResidualTCNBlock(width, int(dilation), dropout)
            for dilation in dilations
        ])
        self.head = nn.Sequential(
            nn.Linear(width * 2, width),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(width, 1),
        )

    def encode(self, values: torch.Tensor) -> torch.Tensor:
        values = self.blocks(self.projection(values))
        return torch.cat([values.mean(dim=-1), values.amax(dim=-1)], dim=1)

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        return self.head(self.encode(values)).squeeze(1)


def _fit_model(
    meta: pd.DataFrame,
    sequences: np.ndarray,
    excluded_countries: Iterable[str],
    epochs: int,
    seed: int,
    device: str | torch.device | None,
    batch_size: int = 512,
    unlabeled_keep_fraction: float = 1.0,
) -> tuple[TinyTemporalTCN, Normalization, dict]:
    excluded = tuple(sorted(set(str(value) for value in excluded_countries)))
    if HOLDOUT in excluded or not set(excluded).issubset(COUNTRIES):
        raise ValueError(f"Invalid country exclusion: {excluded}")
    if epochs < 1 or batch_size < 2:
        raise ValueError("epochs and batch_size must be positive")
    train_mask = _bag_training_mask(
        meta, excluded, unlabeled_keep_fraction, seed
    )
    labels = meta.loc[train_mask, "is_eog_flare"].to_numpy(dtype="float32")
    if len(labels) < 2 or len(np.unique(labels)) != 2:
        raise ValueError(f"Both labels are required after excluding {excluded}")
    training_sequences = sequences[train_mask]
    normalization = _fit_normalization(training_sequences)
    training_sequences = normalization.transform(training_sequences)

    _seed_everything(seed)
    resolved_device = _resolve_device(device)
    model = TinyTemporalTCN().to(resolved_device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=2e-3, weight_decay=1e-3
    )
    positives = float(labels.sum())
    negatives = float(len(labels) - labels.sum())
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(negatives / positives, device=resolved_device)
    )
    dataset = TensorDataset(
        torch.from_numpy(training_sequences), torch.from_numpy(labels)
    )
    generator = torch.Generator().manual_seed(seed)
    training_countries = meta.loc[train_mask, "country"].astype(str).to_numpy()
    country_counts = pd.Series(training_countries).value_counts().sort_index()
    sample_weights = np.asarray(
        [1.0 / country_counts[country] for country in training_countries],
        dtype="float64",
    )
    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(sample_weights),
        num_samples=len(dataset),
        replacement=True,
        generator=generator,
    )
    loader = DataLoader(
        dataset,
        batch_size=min(batch_size, len(dataset)),
        shuffle=False,
        sampler=sampler,
        num_workers=0,
        drop_last=False,
    )
    history = []
    for _ in range(epochs):
        model.train()
        loss_sum = 0.0
        examples = 0
        for batch_values, batch_labels in loader:
            batch_values = batch_values.to(resolved_device)
            batch_labels = batch_labels.to(resolved_device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_values), batch_labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            loss_sum += float(loss.detach().cpu()) * len(batch_values)
            examples += len(batch_values)
        history.append(loss_sum / max(examples, 1))
    diagnostic = {
        "excluded_countries": list(excluded),
        "train_countries": sorted(set(meta.loc[train_mask, "country"])),
        "training_sources": int(train_mask.sum()),
        "training_positives": int(labels.sum()),
        "training_unlabeled": int(len(labels) - labels.sum()),
        "seed": seed,
        "epochs": epochs,
        "batch_size": min(batch_size, len(dataset)),
        "unlabeled_keep_fraction": unlabeled_keep_fraction,
        "device": str(resolved_device),
        "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
        "sampler": {
            "policy": "inverse-country-count",
            "replacement": True,
            "draws_per_epoch": len(dataset),
            "country_source_counts": {
                str(key): int(value) for key, value in country_counts.items()
            },
        },
        "loss": history,
        "normalization": normalization.to_dict(),
    }
    return model, normalization, diagnostic


def _predict(
    model: TinyTemporalTCN,
    normalization: Normalization,
    sequences: np.ndarray,
    device: str | torch.device | None,
    batch_size: int = 512,
    return_embeddings: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    resolved_device = _resolve_device(device)
    model = model.to(resolved_device)
    values = normalization.transform(sequences)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(values)),
        batch_size=min(batch_size, max(len(values), 1)),
        shuffle=False,
        num_workers=0,
    )
    scores = []
    embeddings = []
    model.eval()
    with torch.no_grad():
        for (batch_values,) in loader:
            batch_values = batch_values.to(resolved_device)
            embedding = model.encode(batch_values)
            logit = model.head(embedding).squeeze(1)
            scores.append(torch.sigmoid(logit).cpu().numpy())
            if return_embeddings:
                embeddings.append(embedding.cpu().numpy())
    if not scores:
        return np.empty(0, dtype="float32"), np.empty((0, 64), dtype="float32")
    embedding_array = (
        np.concatenate(embeddings).astype("float32")
        if return_embeddings else np.empty((0, 64), dtype="float32")
    )
    return (
        np.concatenate(scores).astype("float32"),
        embedding_array,
    )


def _fit_bagged_scores(
    meta: pd.DataFrame,
    sequences: np.ndarray,
    query_indices: np.ndarray,
    excluded_countries: Sequence[str],
    epochs: int,
    seed: int,
    device: str | torch.device | None,
    batch_size: int,
    pu_bags: int,
    unlabeled_keep_fraction: float,
) -> tuple[np.ndarray, list[dict]]:
    if pu_bags < 1:
        raise ValueError("pu_bags must be at least one")
    score = np.zeros(len(query_indices), dtype="float64")
    diagnostics = []
    for bag_index in range(pu_bags):
        bag_seed = seed + bag_index * 101
        model, normalization, diagnostic = _fit_model(
            meta, sequences, excluded_countries, epochs, bag_seed, device,
            batch_size, unlabeled_keep_fraction,
        )
        bag_score, _ = _predict(
            model, normalization, sequences[query_indices], device,
            return_embeddings=False,
        )
        score += bag_score / pu_bags
        diagnostic["bag_index"] = bag_index
        diagnostic["pu_bags"] = pu_bags
        diagnostics.append(diagnostic)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return score.astype("float32"), diagnostics


def precompute_nested_scores(
    meta: pd.DataFrame,
    sequences: np.ndarray,
    cohort: pd.DataFrame,
    epochs: int = 24,
    seed: int = 5107,
    device: str | torch.device | None = None,
    batch_size: int = 512,
    pu_bags: int = 2,
    unlabeled_keep_fraction: float = 0.8,
) -> dict:
    """Produce strict one-country population and two-country cohort scores."""
    cohort = _validate_cohort(cohort)
    if len(meta) != len(sequences) or not meta.source_id.is_unique:
        raise ValueError("meta and sequences must be aligned and source IDs unique")
    if HOLDOUT in set(meta.country) or not set(meta.country).issubset(COUNTRIES):
        raise ValueError("Temporal data contains a forbidden country")
    position = pd.Series(np.arange(len(meta), dtype="int64"), index=meta.source_id)
    if not set(cohort.source_id).issubset(position.index):
        raise ValueError("All cohort sources must be present in temporal data")

    single_maps = {}
    pair_maps = {}
    single_frames = []
    pair_frames = []
    diagnostics = []
    population_complete = bool(meta.population_complete.all()) if (
        "population_complete" in meta
    ) else False
    for country_index, country in enumerate(COUNTRIES):
        fold_seed = seed + 1000 * (country_index + 1)
        if population_complete:
            query = meta.loc[
                meta.country.eq(country),
                ["source_id", "country", "block_id", "is_eog_flare", "is_cohort"],
            ].copy()
        else:
            query = cohort.loc[
                cohort.country.eq(country), ["source_id", "country"]
            ].copy()
            query["is_cohort"] = True
        indices = position.loc[query.source_id].to_numpy(dtype="int64")
        score, fold_diagnostics = _fit_bagged_scores(
            meta, sequences, indices, [country], epochs, fold_seed, device,
            batch_size, pu_bags, unlabeled_keep_fraction,
        )
        query["temporal_score"] = score
        query["excluded_country"] = country
        single_frames.append(query)
        cohort_query = query.loc[query.is_cohort]
        single_maps[country] = dict(zip(
            cohort_query.source_id, cohort_query.temporal_score.astype(float)
        ))
        for diagnostic in fold_diagnostics:
            diagnostic["kind"] = "single"
            diagnostic["scored_sources"] = len(query)
            diagnostic["scored_cohort_sources"] = len(cohort_query)
            diagnostic["score_population"] = population_complete
            diagnostic["embedding_dimension"] = 0
            diagnostics.append(diagnostic)

    for pair_index, pair in enumerate(combinations(COUNTRIES, 2)):
        fold_seed = seed + 100000 + 1000 * (pair_index + 1)
        query = cohort.loc[
            cohort.country.isin(pair), ["source_id", "country"]
        ].copy()
        indices = position.loc[query.source_id].to_numpy(dtype="int64")
        score, fold_diagnostics = _fit_bagged_scores(
            meta, sequences, indices, pair, epochs, fold_seed, device,
            batch_size, pu_bags, unlabeled_keep_fraction,
        )
        query["temporal_score"] = score
        query["excluded_country_a"] = pair[0]
        query["excluded_country_b"] = pair[1]
        pair_frames.append(query)
        pair_maps[pair] = dict(zip(query.source_id, score.astype(float)))
        for diagnostic in fold_diagnostics:
            diagnostic["kind"] = "pair"
            diagnostic["scored_sources"] = len(query)
            diagnostic["embedding_dimension"] = 0
            diagnostics.append(diagnostic)

    single_frame = pd.concat(single_frames, ignore_index=True)
    pair_frame = pd.concat(pair_frames, ignore_index=True)
    if not single_frame.source_id.is_unique:
        raise ValueError("Single-country scores must cover each aligned source once")
    expected_single = len(meta) if population_complete else len(cohort)
    if len(single_frame) != expected_single:
        raise ValueError(
            f"Expected {expected_single} strict single-country scores; got {len(single_frame)}"
        )
    return {
        "single_scores": single_maps,
        "pair_scores": pair_maps,
        "single_score_frame": single_frame,
        "pair_score_frame": pair_frame,
        "diagnostics": diagnostics,
        "single_score_population": population_complete,
        "pu_bags": pu_bags,
        "unlabeled_keep_fraction": unlabeled_keep_fraction,
        "protocol": PROTOCOL_VERSION,
    }


def fit_final_tcn(
    meta: pd.DataFrame,
    sequences: np.ndarray,
    output_dir: str | Path,
    epochs: int = 24,
    seed: int = 6101,
    device: str | torch.device | None = None,
    batch_size: int = 512,
    pu_bags: int = 2,
    unlabeled_keep_fraction: float = 0.8,
) -> dict:
    """Fit PU bags on all sampled foreign sources and save artifacts."""
    if pu_bags < 1:
        raise ValueError("pu_bags must be at least one")
    required = {"source_id", "country", "is_eog_flare", "sample_role", "train_selected"}
    missing = required - set(meta.columns)
    if missing:
        raise ValueError(f"Temporal metadata is missing columns: {sorted(missing)}")
    if len(meta) != len(sequences) or not meta.source_id.is_unique:
        raise ValueError("meta and sequences must be aligned with unique source IDs")
    countries = set(meta.country.astype(str))
    if HOLDOUT in countries or not countries.issubset(COUNTRIES):
        raise ValueError("Final TCN training data contains a forbidden country")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    training_signature = hashlib.sha256(
        meta.loc[meta.train_selected, [
            "source_id", "country", "is_eog_flare", "sample_role"
        ]].sort_values(["country", "source_id"]).to_csv(index=False).encode("utf-8")
    ).hexdigest()
    artifact_records = []
    artifact_paths = []
    for bag_index in range(pu_bags):
        bag_seed = seed + bag_index * 101
        model, normalization, diagnostic = _fit_model(
            meta, sequences, [], epochs, bag_seed, device, batch_size,
            unlabeled_keep_fraction,
        )
        artifact_path = output_dir / f"10_final_temporal_tcn_bag{bag_index}.pt"
        artifact = {
            "protocol": PROTOCOL_VERSION,
            "holdout": HOLDOUT,
            "holdout_loaded": False,
            "countries": list(COUNTRIES),
            "window_years": list(WINDOW_YEARS),
            "channels": list(CHANNELS),
            "architecture": DEFAULT_ARCHITECTURE,
            "normalization": normalization.to_dict(),
            "state_dict": {
                key: value.detach().cpu() for key, value in model.state_dict().items()
            },
            "seed": bag_seed,
            "epochs": epochs,
            "bag_index": bag_index,
            "pu_bags": pu_bags,
            "unlabeled_keep_fraction": unlabeled_keep_fraction,
            "training_source_sha256": training_signature,
        }
        torch.save(artifact, artifact_path)
        artifact_hash = _file_hash(artifact_path)
        artifact_paths.append(str(artifact_path))
        artifact_records.append({
            "artifact": artifact_path.name,
            "artifact_sha256": artifact_hash,
            "bag_index": bag_index,
            "seed": bag_seed,
            "training": diagnostic,
        })
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    config = {
        "protocol": PROTOCOL_VERSION,
        "holdout": HOLDOUT,
        "holdout_loaded": False,
        "countries": list(COUNTRIES),
        "window_years": list(WINDOW_YEARS),
        "channels": list(CHANNELS),
        "architecture": DEFAULT_ARCHITECTURE,
        "ensemble": "mean_probability",
        "pu_bags": pu_bags,
        "unlabeled_keep_fraction": unlabeled_keep_fraction,
        "base_seed": seed,
        "epochs": epochs,
        "training_source_sha256": training_signature,
        "artifacts": artifact_records,
    }
    config_path = output_dir / "10_final_temporal_tcn.json"
    config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    return {
        "artifact_path": artifact_paths[0],
        "artifact_paths": artifact_paths,
        "config_path": str(config_path),
        "artifact_sha256": artifact_records[0]["artifact_sha256"],
        "artifact_sha256s": [record["artifact_sha256"] for record in artifact_records],
        "config": config,
    }


def load_tcn_artifact(
    path: str | Path,
    device: str | torch.device | None = None,
) -> tuple[TinyTemporalTCN, Normalization, dict]:
    """Load a saved final model with its exact fold-safe normalization."""
    artifact = torch.load(Path(path), map_location="cpu", weights_only=False)
    if (
        artifact.get("protocol") != PROTOCOL_VERSION
        or artifact.get("holdout_loaded") is not False
        or artifact.get("holdout") != HOLDOUT
    ):
        raise ValueError("Incompatible or holdout-contaminated temporal artifact")
    if artifact.get("channels") != list(CHANNELS):
        raise ValueError("Temporal artifact channel order does not match")
    model = TinyTemporalTCN().to(_resolve_device(device))
    model.load_state_dict(artifact["state_dict"], strict=True)
    model.eval()
    normalizer = Normalization(
        center=np.asarray(artifact["normalization"]["center"], dtype="float32"),
        scale=np.asarray(artifact["normalization"]["scale"], dtype="float32"),
    )
    return model, normalizer, artifact


def score_tcn_artifact(
    path: str | Path,
    sequences: np.ndarray,
    device: str | torch.device | None = None,
    return_embeddings: bool = False,
) -> tuple[np.ndarray, np.ndarray]:
    """Score sequences, optionally returning the 64D pooled embedding."""
    model, normalization, _ = load_tcn_artifact(path, device)
    return _predict(
        model, normalization, sequences, device,
        return_embeddings=return_embeddings,
    )


def score_tcn_ensemble(
    paths: Sequence[str | Path],
    sequences: np.ndarray,
    device: str | torch.device | None = None,
    return_embeddings: bool = False,
) -> tuple[np.ndarray, np.ndarray]:
    """Average saved PU-bag probabilities, optionally averaging embeddings."""
    if not paths:
        raise ValueError("At least one temporal artifact is required")
    score = np.zeros(len(sequences), dtype="float64")
    embedding_sum = None
    for path in paths:
        model, normalization, _ = load_tcn_artifact(path, device)
        bag_score, bag_embedding = _predict(
            model, normalization, sequences, device,
            return_embeddings=return_embeddings,
        )
        score += bag_score / len(paths)
        if return_embeddings:
            if embedding_sum is None:
                embedding_sum = np.zeros_like(bag_embedding, dtype="float64")
            embedding_sum += bag_embedding / len(paths)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    embeddings = (
        embedding_sum.astype("float32")
        if embedding_sum is not None else np.empty((0, 64), dtype="float32")
    )
    return score.astype("float32"), embeddings


In [ ]:
%%writefile /kaggle/working/nb11_code/kg_11_multimodal.py
"""Leakage-safe Sentinel-2, temporal, and FIRMS fusion on foreign countries.

The labelled imagery cohort is too small for end-to-end neural fine-tuning.
This stage therefore freezes a Sentinel-2-pretrained image encoder, uses a
population-trained TCN only through nested country-held-out scores, and keeps
the proven NB9 compact LightGBM branch as the mandatory baseline.
"""
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import platform
import shutil
import time
import urllib.error
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

from kg_08_fusion import (
    BRANCHES as NB9_BRANCHES,
    COUNTRIES,
    HOLDOUT,
    PARAMS as NB9_PARAMS,
    find_unique,
    load_cohort,
    macro_ap,
    macro_f1_threshold,
    metric_row,
)
from kg_10_temporal_tcn import (
    fit_final_tcn,
    precompute_nested_scores,
    prepare_temporal_data,
)


PROTOCOL = "11-multimodal-country-loco-v1"
BASE_COLUMNS = NB9_BRANCHES["early_fusion"]
TCN_STRUCTURED_WEIGHT = 0.80
STAGE1_GATE_FRACTIONS = (0.001, 0.0025, 0.005, 0.01, 0.02)
SSL_WEIGHT_NAME = "SENTINEL2_RGB_MOCO"
SSL_WEIGHT_URL = (
    "https://hf.co/torchgeo/resnet18_sentinel2_rgb_moco/resolve/"
    "e1c032e7785fd0625224cdb6699aa138bb304eec/"
    "resnet18_sentinel2_rgb_moco-e3a335e3.pth"
)
VARIANTS = {
    "nb9_baseline": {"temporal": False, "ssl": False},
    "temporal_features": {"temporal": True, "ssl": False},
    "ssl_features": {"temporal": False, "ssl": True},
    "structured_full": {"temporal": True, "ssl": True},
}
FORBIDDEN_FEATURE_COLUMNS = {
    "country", "latitude", "longitude", "lat", "lon", "type",
    "eog_dist_m", "eog_flare_id", "is_eog_flare", "block_id",
}
# Keep the reference branch exactly aligned with the completed NB9 experiment.
PARAMS = dict(NB9_PARAMS)


class OptionalSSLUnavailable(RuntimeError):
    """Raised only when the optional SSL dependency or checkpoint is unavailable."""


def file_hash(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def package_version(package: str) -> str | None:
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None


def _cohort_hash(cohort: pd.DataFrame) -> str:
    value = cohort[["source_id", "country", "is_eog_flare"]].sort_values(
        "source_id"
    ).to_csv(index=False)
    return hashlib.sha256(value.encode()).hexdigest()


def _fill_rgb(image: np.ndarray) -> np.ndarray:
    """Return finite Sentinel-2 RGB reflectance in B4, B3, B2 order."""
    rgb = image[[2, 1, 0]].astype("float32", copy=True)
    for band in range(3):
        finite = np.isfinite(rgb[band])
        fill = float(np.median(rgb[band, finite])) if finite.any() else 0.0
        rgb[band, ~finite] = fill
    return np.clip(rgb, -0.05, 1.0)


def extract_ssl_embeddings(
    input_root: str | Path,
    cohort: pd.DataFrame,
    cache_path: str | Path,
    batch_size: int = 24,
) -> tuple[pd.DataFrame, dict]:
    """Extract frozen full-chip and 1 km center embeddings on GPU when present."""
    import torch

    cache_path = Path(cache_path)
    manifest_path = cache_path.with_suffix(".json")
    expected = {
        "cohort_sha256": _cohort_hash(cohort),
        "weight_name": SSL_WEIGHT_NAME,
        "weight_url": SSL_WEIGHT_URL,
        "views": ["full_2km", "center_1km"],
        "bands": ["B4", "B3", "B2"],
    }
    if cache_path.exists() and manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if all(manifest.get(key) == value for key, value in expected.items()):
            frame = pd.read_parquet(cache_path)
            if frame.source_id.is_unique and set(frame.source_id) == set(cohort.source_id):
                return frame, manifest

    try:
        from torchgeo.models import ResNet18_Weights, resnet18
    except (ImportError, OSError) as error:
        raise OptionalSSLUnavailable(
            "Install torchgeo==0.7.1 and timm before extracting SSL embeddings"
        ) from error

    weights = ResNet18_Weights.SENTINEL2_RGB_MOCO
    weight_bands = [
        str(band).upper().split(".")[-1].replace("B0", "B")
        for band in weights.meta.get("bands", [])
    ]
    if weight_bands != ["B4", "B3", "B2"]:
        raise ValueError(f"Unexpected pretrained band order: {weights.meta}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    try:
        model = resnet18(weights=weights)
    except (urllib.error.URLError, TimeoutError, ConnectionError) as error:
        raise OptionalSSLUnavailable(
            "The optional TorchGeo checkpoint could not be downloaded"
        ) from error
    model.fc = torch.nn.Identity()
    model = model.to(device).eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    transform = weights.transforms()
    with torch.inference_mode():
        probe = transform(torch.zeros((3, 200, 200), dtype=torch.float32))
        probe_embedding = model(probe.unsqueeze(0).to(device))
    if tuple(probe_embedding.shape) != (1, 512):
        raise ValueError(
            f"TorchGeo encoder smoke test returned {tuple(probe_embedding.shape)}"
        )
    del probe, probe_embedding

    sample_path = find_unique(Path(input_root), "pilot_sources.csv")
    chip_root = sample_path.parent
    sample = pd.read_csv(sample_path, usecols=["source_id", "chip_id"])
    mapping = sample.set_index("source_id").chip_id
    if not set(cohort.source_id).issubset(mapping.index):
        raise ValueError("Cohort source is absent from the frozen chip manifest")

    rows = []
    tensors = []
    source_ids = []

    def flush() -> None:
        if not tensors:
            return
        batch = torch.stack(tensors).to(device, non_blocking=True)
        with torch.inference_mode():
            if device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    embedding = model(batch)
            else:
                embedding = model(batch)
        values = embedding.float().cpu().numpy()
        for source_id, value in zip(source_ids, values):
            rows.append((source_id, value))
        tensors.clear()
        source_ids.clear()

    for source_id in cohort.source_id:
        chip_id = mapping.loc[source_id]
        path = chip_root / f"{chip_id}.npz"
        if not path.exists():
            raise FileNotFoundError(f"Missing successful chip: {path}")
        with np.load(path, allow_pickle=False) as data:
            rgb = _fill_rgb(data["reflectance"])
        full = transform(torch.from_numpy(rgb * 10_000.0))
        center = transform(torch.from_numpy(rgb[:, 50:150, 50:150] * 10_000.0))
        tensors.extend([full, center])
        source_ids.extend([f"{source_id}|full", f"{source_id}|center"])
        if len(tensors) >= 2 * batch_size:
            flush()
    flush()

    by_key = {key: value for key, value in rows}
    matrix = []
    for source_id in cohort.source_id:
        matrix.append(np.concatenate([
            by_key[f"{source_id}|full"], by_key[f"{source_id}|center"]
        ]).astype("float32"))
    matrix = np.stack(matrix)
    if not np.isfinite(matrix).all() or matrix.shape[1] != 1024:
        raise ValueError(f"Invalid SSL embedding matrix: {matrix.shape}")
    columns = [f"ssl_{index:04d}" for index in range(matrix.shape[1])]
    frame = pd.DataFrame(matrix, columns=columns)
    frame.insert(0, "source_id", cohort.source_id.to_numpy())
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(cache_path, index=False)
    manifest = {
        **expected,
        "n_sources": len(frame),
        "n_features": len(columns),
        "device": str(device),
        "torch": torch.__version__,
        "torchgeo": importlib.metadata.version("torchgeo"),
        "embedding_sha256": file_hash(cache_path),
    }
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return frame, manifest


def _aligned_extra(
    cohort: pd.DataFrame,
    descriptors: pd.DataFrame,
    embeddings: pd.DataFrame,
) -> tuple[pd.DataFrame, np.ndarray, list[str]]:
    descriptor_columns = [column for column in descriptors if column.startswith("ts_")]
    if not descriptor_columns:
        raise ValueError("No explicit ts_* temporal descriptors were supplied")
    forbidden = FORBIDDEN_FEATURE_COLUMNS & set(descriptor_columns)
    eog_columns = [column for column in descriptor_columns if "eog" in column.lower()]
    if forbidden or eog_columns:
        raise ValueError(
            f"Forbidden temporal descriptor: {sorted(forbidden | set(eog_columns))}"
        )
    descriptor_frame = cohort[["source_id"]].merge(
        descriptors[["source_id"] + descriptor_columns],
        on="source_id", how="left", validate="one_to_one",
    )
    if descriptor_frame[descriptor_columns].isna().all(axis=1).any():
        raise ValueError("At least one cohort row has no temporal descriptors")
    embedding_columns = [column for column in embeddings if column.startswith("ssl_")]
    embedding_frame = cohort[["source_id"]].merge(
        embeddings, on="source_id", how="left", validate="one_to_one"
    )
    embedding_matrix = embedding_frame[embedding_columns].to_numpy(dtype="float32")
    if not np.isfinite(embedding_matrix).all():
        raise ValueError("Non-finite frozen image embedding")
    return descriptor_frame, embedding_matrix, descriptor_columns


def make_matrices(
    cohort: pd.DataFrame,
    descriptor_frame: pd.DataFrame,
    descriptor_columns: list[str],
    embedding_matrix: np.ndarray,
    fit_index: np.ndarray,
    test_index: np.ndarray,
    variant: str,
    pca_components: int,
):
    config = VARIANTS[variant]
    names = list(BASE_COLUMNS)
    fit_parts = [cohort.iloc[fit_index][BASE_COLUMNS].to_numpy(dtype="float32")]
    test_parts = [cohort.iloc[test_index][BASE_COLUMNS].to_numpy(dtype="float32")]
    if config["temporal"]:
        fit_parts.append(
            descriptor_frame.iloc[fit_index][descriptor_columns].to_numpy(dtype="float32")
        )
        test_parts.append(
            descriptor_frame.iloc[test_index][descriptor_columns].to_numpy(dtype="float32")
        )
        names.extend(descriptor_columns)
    transformer = None
    if config["ssl"]:
        components = min(pca_components, len(fit_index) - 1, embedding_matrix.shape[1])
        if components < 2:
            raise ValueError("Insufficient training rows for fold-local SSL PCA")
        scaler = StandardScaler()
        fit_scaled = scaler.fit_transform(embedding_matrix[fit_index])
        pca = PCA(n_components=components, whiten=True, random_state=0)
        fit_parts.append(pca.fit_transform(fit_scaled).astype("float32"))
        test_parts.append(
            pca.transform(scaler.transform(embedding_matrix[test_index])).astype("float32")
        )
        names.extend([f"ssl_pc_{index:02d}" for index in range(components)])
        transformer = {"scaler": scaler, "pca": pca}
    return np.concatenate(fit_parts, axis=1), np.concatenate(test_parts, axis=1), names, transformer


def train_model(x, y, seed, rounds):
    params = dict(PARAMS)
    params.update({key: seed for key in [
        "seed", "feature_fraction_seed", "bagging_seed", "data_random_seed"
    ]})
    return lgb.train(
        params,
        lgb.Dataset(x, label=y, free_raw_data=True),
        num_boost_round=rounds,
    )


def structured_inner_oof(
    train: pd.DataFrame,
    descriptor_frame: pd.DataFrame,
    descriptor_columns: list[str],
    embedding_matrix: np.ndarray,
    variant: str,
    seed: int,
    rounds: int,
    pca_components: int,
) -> np.ndarray:
    score = np.empty(len(train), dtype="float64")
    for inner_index, country in enumerate(sorted(train.country.unique())):
        fit_index = np.flatnonzero(train.country.ne(country).to_numpy())
        test_index = np.flatnonzero(train.country.eq(country).to_numpy())
        x_fit, x_test, _, _ = make_matrices(
            train, descriptor_frame, descriptor_columns, embedding_matrix,
            fit_index, test_index, variant, pca_components,
        )
        model = train_model(
            x_fit,
            train.iloc[fit_index].is_eog_flare.to_numpy(dtype="int8"),
            seed + inner_index * 100,
            rounds,
        )
        score[test_index] = model.predict(x_test)
    return score


def structured_outer_predict(
    train: pd.DataFrame,
    test: pd.DataFrame,
    train_descriptors: pd.DataFrame,
    test_descriptors: pd.DataFrame,
    descriptor_columns: list[str],
    train_embeddings: np.ndarray,
    test_embeddings: np.ndarray,
    variant: str,
    seeds: list[int],
    rounds: int,
    pca_components: int,
) -> np.ndarray:
    combined = pd.concat([train, test], ignore_index=True)
    combined_descriptors = pd.concat(
        [train_descriptors, test_descriptors], ignore_index=True
    )
    combined_embeddings = np.concatenate([train_embeddings, test_embeddings], axis=0)
    fit_index = np.arange(len(train))
    test_index = np.arange(len(train), len(combined))
    x_fit, x_test, _, _ = make_matrices(
        combined, combined_descriptors, descriptor_columns, combined_embeddings,
        fit_index, test_index, variant, pca_components,
    )
    score = np.zeros(len(test), dtype="float64")
    y = train.is_eog_flare.to_numpy(dtype="int8")
    for model_seed in seeds:
        score += train_model(x_fit, y, model_seed, rounds).predict(x_test) / len(seeds)
    return score


def _lookup_score(mapping, country: str, source_ids: pd.Series) -> np.ndarray:
    values = mapping[country]
    if isinstance(values, pd.DataFrame):
        values = values.set_index("source_id").score
    elif isinstance(values, dict):
        values = pd.Series(values)
    if not isinstance(values, pd.Series):
        raise TypeError(f"Unexpected TCN score mapping for {country}: {type(values)}")
    missing = set(source_ids) - set(values.index)
    if missing:
        raise ValueError(f"Missing {len(missing)} TCN scores for {country}")
    score = values.loc[source_ids].to_numpy(dtype="float64")
    if not np.isfinite(score).all():
        raise ValueError("Non-finite TCN score")
    return score


def _lookup_flat_score(mapping, source_ids: pd.Series) -> np.ndarray:
    if isinstance(mapping, pd.DataFrame):
        values = mapping.set_index("source_id").score
    elif isinstance(mapping, dict):
        values = pd.Series(mapping, dtype="float64")
    elif isinstance(mapping, pd.Series):
        values = mapping
    else:
        raise TypeError(f"Unexpected flat TCN score mapping: {type(mapping)}")
    missing = set(source_ids) - set(values.index)
    if missing:
        raise ValueError(f"Missing {len(missing)} nested TCN scores")
    score = values.loc[source_ids].to_numpy(dtype="float64")
    if not np.isfinite(score).all():
        raise ValueError("Non-finite nested TCN score")
    return score


def _pair_mapping(pair_scores, first: str, second: str):
    key = tuple(sorted((first, second)))
    if key not in pair_scores:
        raise KeyError(f"Missing nested TCN pair scores for {key}")
    return pair_scores[key]


def _country_ap_values(frame: pd.DataFrame, score: np.ndarray) -> dict[str, float]:
    values = {}
    for country, part in frame.groupby("country"):
        values[country] = macro_ap(part.reset_index(drop=True), score[part.index])
    return values


def country_percentile(frame: pd.DataFrame, score: np.ndarray) -> np.ndarray:
    """Convert scores to within-country percentile ranks without labels."""
    if len(frame) != len(score):
        raise ValueError("Frame and score lengths differ")
    ranked = np.empty(len(frame), dtype="float64")
    for _, part in frame.groupby("country", sort=False):
        indices = part.index.to_numpy(dtype="int64")
        ranked[indices] = pd.Series(score[indices]).rank(
            method="average", pct=True
        ).to_numpy(dtype="float64")
    return ranked


def candidate_scores(
    frame: pd.DataFrame,
    structured_scores: dict[str, np.ndarray],
    tcn_score: np.ndarray,
) -> dict[str, np.ndarray]:
    """Build fixed candidates, including conservative rank-level blends."""
    candidates = {key: np.asarray(value) for key, value in structured_scores.items()}
    candidates["tcn_only"] = np.asarray(tcn_score)
    tcn_rank = country_percentile(frame, tcn_score)
    for variant, score in structured_scores.items():
        key = f"rank_blend_{variant}"
        candidates[key] = (
            TCN_STRUCTURED_WEIGHT * country_percentile(frame, score)
            + (1.0 - TCN_STRUCTURED_WEIGHT) * tcn_rank
        )
    return candidates


def choose_inner_candidate(
    frame: pd.DataFrame,
    candidates: dict[str, np.ndarray],
    minimum_improved_countries: int,
    maximum_worst_drop: float,
) -> tuple[str, np.ndarray, list[dict]]:
    """Apply a predeclared baseline guard using training countries only."""
    baseline = "nb9_baseline"
    baseline_country = _country_ap_values(frame, candidates[baseline])
    baseline_ap = float(np.mean(list(baseline_country.values())))
    rows = []
    for branch, score in candidates.items():
        by_country = _country_ap_values(frame, score)
        deltas = np.asarray([
            by_country[country] - baseline_country[country]
            for country in sorted(by_country)
        ])
        rows.append({
            "branch": branch,
            "macro_ap": float(np.mean(list(by_country.values()))),
            "ap_gain_vs_baseline": float(np.mean(deltas)),
            "improved_countries": int((deltas > 0).sum()),
            "worst_ap_delta": float(deltas.min()),
            "eligible": bool(
                branch == baseline
                or (
                    np.mean(list(by_country.values())) >= baseline_ap + 0.005
                    and (deltas > 0).sum() >= minimum_improved_countries
                    and deltas.min() >= -maximum_worst_drop
                )
            ),
        })
    table = pd.DataFrame(rows)
    selected = table.loc[table.eligible].sort_values(
        ["macro_ap", "branch"], ascending=[False, True]
    ).iloc[0]
    name = str(selected.branch)
    return name, candidates[name], table.sort_values(
        ["eligible", "macro_ap"], ascending=False
    ).to_dict("records")


def _global_branch_selection(metrics: pd.DataFrame) -> tuple[str, pd.DataFrame]:
    summary = metrics.groupby("branch").agg(
        macro_f1=("f1", "mean"),
        macro_pr_auc=("pr_auc", "mean"),
        macro_roc_auc=("roc_auc", "mean"),
        worst_country_pr_auc=("pr_auc", "min"),
    ).reset_index()
    baseline = metrics.loc[metrics.branch.eq("nb9_baseline")].set_index("country")
    checks = []
    baseline_ap = float(baseline.pr_auc.mean())
    for branch in summary.branch:
        current = metrics.loc[metrics.branch.eq(branch)].set_index("country")
        delta = current.pr_auc - baseline.pr_auc
        checks.append({
            "branch": branch,
            "ap_gain_vs_baseline": float(current.pr_auc.mean() - baseline_ap),
            "improved_countries": int((delta > 0).sum()),
            "worst_ap_delta": float(delta.min()),
            "eligible": bool(
                branch == "nb9_baseline"
                or (
                    current.pr_auc.mean() >= baseline_ap + 0.005
                    and (delta > 0).sum() >= 4
                    and delta.min() >= -0.03
                )
            ),
        })
    checks = pd.DataFrame(checks)
    summary = summary.merge(checks, on="branch", validate="one_to_one")
    eligible = summary.loc[summary.eligible].sort_values(
        ["macro_pr_auc", "macro_f1"], ascending=False
    )
    return str(eligible.iloc[0].branch), summary.sort_values(
        ["eligible", "macro_pr_auc"], ascending=False
    )


def load_stage1(input_root: str | Path) -> tuple[pd.DataFrame, dict, list[Path]]:
    """Load the frozen NB8 population branch without retraining it."""
    root = Path(input_root)
    manifest_path = find_unique(root, "05e_manifest.json")
    prediction_path = find_unique(root, "05e_loco_predictions.parquet")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if (
        manifest.get("protocol") != "05e-domain-revamp-v1"
        or manifest.get("status") != "complete"
        or manifest.get("holdout_country") != HOLDOUT
        or manifest.get("holdout_loaded") is not False
    ):
        raise ValueError("NB8 population manifest is incompatible or contaminated")
    columns = [
        "source_id", "country", "block_id", "is_eog_flare",
        "eog_flare_id", "score", "model_variant",
    ]
    frame = pd.read_parquet(prediction_path, columns=columns)
    frame["source_id"] = frame.source_id.astype(str)
    frame["country"] = frame.country.astype(str)
    if (
        len(frame) != int(manifest.get("n_sources", -1))
        or not frame.source_id.is_unique
        or set(frame.country) != set(COUNTRIES)
        or HOLDOUT in set(frame.country)
    ):
        raise ValueError("NB8 population predictions are incomplete or invalid")
    model_paths = sorted(manifest_path.parent.parent.rglob("05e_final_model_*.txt"))
    if len(model_paths) != 3:
        raise FileNotFoundError(f"Expected three frozen NB8 models; found {model_paths}")
    metadata = {
        "manifest_path": str(manifest_path),
        "prediction_path": str(prediction_path),
        "manifest_sha256": file_hash(manifest_path),
        "prediction_sha256": file_hash(prediction_path),
        "protocol": manifest["protocol"],
        "n_sources": len(frame),
        "n_positive": int(frame.is_eog_flare.sum()),
        "original_macro_pr_auc": manifest.get("macro_loco_pr_auc"),
        "original_macro_f1": manifest.get("macro_loco_f1"),
        "selected_features": manifest.get("selected_features"),
        "selected_variant": manifest.get("selected_variant"),
        "final_threshold": manifest.get("final_threshold"),
        "input_sha256": manifest.get("input_sha256", {}),
    }
    return frame, metadata, model_paths


def evaluate_stage1(
    nb8: pd.DataFrame,
    temporal_frame: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, str, dict]:
    """Evaluate the population TCN and a fixed rank blend at real prevalence."""
    required_temporal = {
        "source_id", "country", "is_eog_flare", "temporal_score"
    }
    if not required_temporal.issubset(temporal_frame):
        raise ValueError("Full-population temporal scores are required for Stage A")
    temporal = temporal_frame[list(required_temporal)].copy()
    frame = nb8.merge(
        temporal,
        on=["source_id", "country"],
        how="inner",
        validate="one_to_one",
        suffixes=("_nb8", "_tcn"),
    )
    if len(frame) != len(nb8):
        raise ValueError("TCN scores do not cover the complete NB8 population")
    if not frame.is_eog_flare_nb8.astype("int8").equals(
        frame.is_eog_flare_tcn.astype("int8")
    ):
        raise ValueError("NB8 and TCN population labels disagree")
    frame = frame.rename(columns={
        "is_eog_flare_nb8": "is_eog_flare",
        "score": "nb8_score",
    }).drop(columns=["is_eog_flare_tcn"])
    frame["nb8_rank"] = frame.groupby("country", observed=True).nb8_score.rank(
        method="average", pct=True
    )
    frame["tcn_rank"] = frame.groupby("country", observed=True).temporal_score.rank(
        method="average", pct=True
    )
    frame["rank_blend_score"] = (
        TCN_STRUCTURED_WEIGHT * frame.nb8_rank
        + (1.0 - TCN_STRUCTURED_WEIGHT) * frame.tcn_rank
    )
    branches = {
        "nb8_population": "nb8_score",
        "tcn_population": "temporal_score",
        "fixed_rank_blend": "rank_blend_score",
    }
    metric_rows = []
    gate_rows = []
    for country, part in frame.groupby("country", observed=True):
        y = part.is_eog_flare.to_numpy(dtype="int8")
        total_sites = part.loc[part.is_eog_flare.eq(1), "eog_flare_id"].nunique()
        for branch, column in branches.items():
            score = part[column].to_numpy(dtype="float64")
            metric_rows.append({
                "country": country,
                "branch": branch,
                "n": len(part),
                "n_positive": int(y.sum()),
                "pr_auc": average_precision_score(y, score),
                "roc_auc": roc_auc_score(y, score),
            })
            order = np.argsort(score)[::-1]
            for fraction in STAGE1_GATE_FRACTIONS:
                count = max(1, int(np.ceil(fraction * len(part))))
                selected = part.iloc[order[:count]]
                true_positive = int(selected.is_eog_flare.sum())
                hit_sites = selected.loc[
                    selected.is_eog_flare.eq(1), "eog_flare_id"
                ].nunique()
                gate_rows.append({
                    "country": country,
                    "branch": branch,
                    "top_fraction": fraction,
                    "candidates": count,
                    "precision_at_gate": true_positive / count,
                    "source_recall_at_gate": true_positive / max(int(y.sum()), 1),
                    "site_recall_at_gate": hit_sites / max(total_sites, 1),
                })
    metrics = pd.DataFrame(metric_rows)
    gates = pd.DataFrame(gate_rows)
    baseline = metrics.loc[metrics.branch.eq("nb8_population")].set_index("country")
    blend = metrics.loc[metrics.branch.eq("fixed_rank_blend")].set_index("country")
    delta = blend.pr_auc - baseline.pr_auc
    guard = {
        "macro_ap_gain": float(delta.mean()),
        "improved_countries": int((delta > 0).sum()),
        "worst_country_ap_delta": float(delta.min()),
        "minimum_macro_ap_gain": 0.005,
        "minimum_improved_countries": 4,
        "maximum_worst_country_ap_drop": 0.03,
    }
    selected_branch = "fixed_rank_blend" if (
        guard["macro_ap_gain"] >= guard["minimum_macro_ap_gain"]
        and guard["improved_countries"] >= guard["minimum_improved_countries"]
        and guard["worst_country_ap_delta"]
        >= -guard["maximum_worst_country_ap_drop"]
    ) else "nb8_population"
    frame["selected_stage1_score"] = frame[
        branches[selected_branch]
    ]
    return frame, metrics, gates, selected_branch, guard


def validate_stage1_feature_hashes(stage1_metadata: dict, temporal_metadata: dict) -> None:
    """Require NB8 predictions and the current NB2 features to share a lineage."""
    for country in COUNTRIES:
        name = f"features_{country}_2022_2024.parquet"
        nb8_hash = stage1_metadata.get("input_sha256", {}).get(name)
        current_hash = temporal_metadata.get("input_sha256", {}).get(name)
        if not nb8_hash or nb8_hash != current_hash:
            raise ValueError(f"NB8 and NB2 use different feature data for {name}")


def run(
    input_root="/kaggle/input",
    output_root="/kaggle/working/nb11_multimodal",
    rounds=300,
    tcn_epochs=8,
    negative_ratio=10,
    pu_bags=2,
    tcn_batch_size=512,
    pca_components=16,
    seed=131,
    include_ssl=True,
):
    started = time.time()
    input_root = Path(input_root)
    output_root = Path(output_root)
    outputs = output_root / "outputs"
    cache = output_root / "cache"
    models = output_root / "models"
    for directory in [outputs, cache, models]:
        directory.mkdir(parents=True, exist_ok=True)
    manifest_path = outputs / "11_manifest.json"
    if manifest_path.exists():
        raise FileExistsError("Use a fresh output directory for a complete run")
    shutil.copy2(Path(__file__), output_root / Path(__file__).name)
    for dependency in ["kg_08_fusion.py", "kg_10_temporal_tcn.py"]:
        source = Path(__file__).with_name(dependency)
        if not source.exists():
            raise FileNotFoundError(f"Missing bundled implementation: {source}")
        shutil.copy2(source, output_root / dependency)

    cohort, nb9_metadata = load_cohort(input_root)
    if HOLDOUT in set(cohort.country):
        raise ValueError("India is forbidden during multimodal model selection")
    nb8_predictions, stage1_metadata, stage1_model_paths = load_stage1(input_root)
    available_variants = list(VARIANTS)
    if include_ssl:
        try:
            embeddings, ssl_metadata = extract_ssl_embeddings(
                input_root, cohort, cache / "11_ssl_embeddings.parquet"
            )
            ssl_metadata["status"] = "available"
        except OptionalSSLUnavailable as error:
            embeddings = pd.DataFrame({"source_id": cohort.source_id})
            available_variants = [
                variant for variant in available_variants
                if not VARIANTS[variant]["ssl"]
            ]
            ssl_metadata = {
                "status": "disabled",
                "reason": f"{type(error).__name__}: {error}",
                "weight_name": SSL_WEIGHT_NAME,
                "weight_url": SSL_WEIGHT_URL,
            }
            print(f"Optional SSL branch disabled: {ssl_metadata['reason']}", flush=True)
    else:
        embeddings = pd.DataFrame({"source_id": cohort.source_id})
        available_variants = [
            variant for variant in available_variants
            if not VARIANTS[variant]["ssl"]
        ]
        ssl_metadata = {
            "status": "disabled",
            "reason": "disabled by configuration",
            "weight_name": SSL_WEIGHT_NAME,
            "weight_url": SSL_WEIGHT_URL,
        }
    temporal_meta, sequences, descriptors, temporal_metadata = prepare_temporal_data(
        input_root,
        cohort,
        negative_ratio=negative_ratio,
        seed=seed,
        include_population=True,
    )
    validate_stage1_feature_hashes(stage1_metadata, temporal_metadata)
    descriptor_frame, embedding_matrix, descriptor_columns = _aligned_extra(
        cohort, descriptors, embeddings
    )

    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    temporal_scores = precompute_nested_scores(
        temporal_meta, sequences, cohort,
        epochs=tcn_epochs,
        seed=seed,
        device=device,
        batch_size=tcn_batch_size,
        pu_bags=pu_bags,
        unlabeled_keep_fraction=0.8,
    )
    single_scores = temporal_scores["single_scores"]
    pair_scores = temporal_scores["pair_scores"]
    stage1_predictions, stage1_metrics, stage1_gates, stage1_selected, stage1_guard = (
        evaluate_stage1(
            nb8_predictions,
            temporal_scores["single_score_frame"],
        )
    )

    metric_rows = []
    prediction_parts = []
    selection_rows = []
    for outer_index, held_out in enumerate(COUNTRIES):
        train_mask = cohort.country.ne(held_out).to_numpy()
        test_mask = ~train_mask
        train = cohort.loc[train_mask].reset_index(drop=True)
        test = cohort.loc[test_mask].reset_index(drop=True)
        train_descriptors = descriptor_frame.loc[train_mask].reset_index(drop=True)
        test_descriptors = descriptor_frame.loc[test_mask].reset_index(drop=True)
        train_embeddings = embedding_matrix[train_mask]
        test_embeddings = embedding_matrix[test_mask]

        inner_scores = {}
        outer_scores = {}
        for variant in available_variants:
            inner_scores[variant] = structured_inner_oof(
                train, train_descriptors, descriptor_columns, train_embeddings,
                variant, seed + outer_index * 1000, rounds, pca_components,
            )
            threshold, inner_f1 = macro_f1_threshold(train, inner_scores[variant])
            outer_scores[variant] = structured_outer_predict(
                train, test, train_descriptors, test_descriptors,
                descriptor_columns, train_embeddings, test_embeddings,
                variant,
                [seed + 20_000 + outer_index * 1000 + offset for offset in (0, 101, 202)],
                rounds, pca_components,
            )
            metric_rows.append(metric_row(
                test, outer_scores[variant], threshold, variant, held_out
            ))
            selection_rows.append({
                "held_out_country": held_out,
                "branch": variant,
                "inner_macro_ap": macro_ap(train, inner_scores[variant]),
                "inner_macro_f1": inner_f1,
                "threshold": threshold,
            })

        tcn_inner = np.empty(len(train), dtype="float64")
        for inner_country in sorted(train.country.unique()):
            mask = train.country.eq(inner_country)
            mapping = _pair_mapping(pair_scores, held_out, inner_country)
            tcn_inner[mask] = _lookup_flat_score(
                mapping, train.loc[mask, "source_id"]
            )
        tcn_outer = _lookup_score(single_scores, held_out, test.source_id)
        inner_candidates = candidate_scores(train, inner_scores, tcn_inner)
        outer_candidates = candidate_scores(test, outer_scores, tcn_outer)
        for branch in sorted(set(inner_candidates) - set(VARIANTS)):
            threshold, inner_f1 = macro_f1_threshold(train, inner_candidates[branch])
            metric_rows.append(metric_row(
                test, outer_candidates[branch], threshold, branch, held_out
            ))
            selection_rows.append({
                "held_out_country": held_out,
                "branch": branch,
                "inner_macro_ap": macro_ap(train, inner_candidates[branch]),
                "inner_macro_f1": inner_f1,
                "threshold": threshold,
            })

        champion, champion_inner, diagnostics = choose_inner_candidate(
            train,
            inner_candidates,
            minimum_improved_countries=3,
            maximum_worst_drop=0.04,
        )
        champion_threshold, champion_inner_f1 = macro_f1_threshold(
            train, champion_inner
        )
        champion_outer = outer_candidates[champion]
        metric_rows.append(metric_row(
            test, champion_outer, champion_threshold, "nested_champion", held_out
        ))
        selection_rows.append({
            "held_out_country": held_out,
            "branch": "nested_champion",
            "selected_candidate": champion,
            "inner_macro_ap": macro_ap(train, champion_inner),
            "inner_macro_f1": champion_inner_f1,
            "threshold": champion_threshold,
            "candidate_diagnostics": json.dumps(diagnostics),
        })

        part = test[[
            "source_id", "country", "is_eog_flare", "eog_flare_id", "block_id"
        ]].copy()
        for branch, score in outer_candidates.items():
            part[f"score_{branch}"] = score
        part["score_nested_champion"] = champion_outer
        part["nested_selected_candidate"] = champion
        prediction_parts.append(part)
        print(f"Completed multimodal holdout {held_out}", flush=True)

    metrics = pd.DataFrame(metric_rows)
    predictions = pd.concat(prediction_parts, ignore_index=True)
    selections = pd.DataFrame(selection_rows)
    summary = metrics.groupby("branch").agg(
        macro_f1=("f1", "mean"),
        macro_pr_auc=("pr_auc", "mean"),
        macro_roc_auc=("roc_auc", "mean"),
        worst_country_pr_auc=("pr_auc", "min"),
    ).reset_index().sort_values(
        ["macro_pr_auc", "macro_f1"], ascending=False
    )
    ordered = predictions.set_index("source_id").loc[cohort.source_id]
    tcn_oof = ordered.score_tcn_only.to_numpy()
    structured_oof = {
        variant: ordered[f"score_{variant}"].to_numpy()
        for variant in available_variants
    }
    global_candidates = candidate_scores(cohort, structured_oof, tcn_oof)
    selected, final_score, final_diagnostics = choose_inner_candidate(
        cohort,
        global_candidates,
        minimum_improved_countries=4,
        maximum_worst_drop=0.03,
    )
    if selected == "tcn_only":
        final_weight = 0.0
        model_variant = None
    elif selected.startswith("rank_blend_"):
        model_variant = selected.removeprefix("rank_blend_")
        final_weight = TCN_STRUCTURED_WEIGHT
    else:
        final_weight = None
        model_variant = selected
    final_threshold, _ = macro_f1_threshold(cohort, final_score)
    final_score_lookup = dict(zip(cohort.source_id, final_score))
    predictions["score_final_selected"] = predictions.source_id.map(final_score_lookup)

    model_artifacts = []
    importance = pd.DataFrame()
    feature_names: list[str] = []
    if model_variant is not None:
        all_index = np.arange(len(cohort))
        x, _, feature_names, transformer = make_matrices(
            cohort, descriptor_frame, descriptor_columns, embedding_matrix,
            all_index, all_index, model_variant, pca_components,
        )
        gain = np.zeros(len(feature_names))
        for model_index, final_seed in enumerate(
            [seed + 30_000 + offset for offset in (0, 101, 202)]
        ):
            model = train_model(
                x, cohort.is_eog_flare.to_numpy(dtype="int8"), final_seed, rounds
            )
            path = models / f"{model_variant}_{model_index}.txt"
            model.save_model(str(path))
            model_artifacts.append(path.name)
            gain += model.feature_importance("gain") / 3
        importance = pd.DataFrame({"feature": feature_names, "mean_gain": gain})
        importance.sort_values("mean_gain", ascending=False).to_csv(
            outputs / "11_final_feature_importance.csv", index=False
        )
        if transformer is not None:
            joblib.dump(transformer, models / "ssl_transformer.joblib")
            model_artifacts.append("ssl_transformer.joblib")

    tcn_artifact = fit_final_tcn(
        temporal_meta, sequences, models,
        epochs=tcn_epochs,
        seed=seed + 40_000,
        device=device,
        batch_size=tcn_batch_size,
        pu_bags=pu_bags,
        unlabeled_keep_fraction=0.8,
    )
    tcn_artifact["artifact_path"] = Path(tcn_artifact["artifact_path"]).name
    tcn_artifact["artifact_paths"] = [
        Path(path).name for path in tcn_artifact["artifact_paths"]
    ]
    tcn_artifact["config_path"] = Path(tcn_artifact["config_path"]).name
    stage1_model_dir = models / "stage1_nb8"
    stage1_model_dir.mkdir(parents=True, exist_ok=True)
    copied_stage1_models = []
    for source_path in stage1_model_paths:
        target = stage1_model_dir / source_path.name
        shutil.copy2(source_path, target)
        copied_stage1_models.append(str(target.relative_to(models)))
    metrics.to_csv(outputs / "11_country_metrics.csv", index=False)
    summary.to_csv(outputs / "11_branch_summary.csv", index=False)
    selections.to_csv(outputs / "11_inner_selection.csv", index=False)
    predictions.to_parquet(outputs / "11_loco_predictions.parquet", index=False)
    stage1_predictions.to_parquet(
        outputs / "11_stage1_population_predictions.parquet", index=False
    )
    stage1_metrics.to_csv(outputs / "11_stage1_country_metrics.csv", index=False)
    stage1_gates.to_csv(outputs / "11_stage1_gate_metrics.csv", index=False)
    descriptor_keep = temporal_meta.train_selected | temporal_meta.is_cohort
    descriptors.loc[descriptor_keep].to_parquet(
        cache / "11_temporal_descriptors.parquet", index=False
    )
    pd.DataFrame(temporal_scores["diagnostics"]).to_csv(
        outputs / "11_tcn_training_diagnostics.csv", index=False
    )
    selected_schema = {
        "stage1_branch": stage1_selected,
        "stage1_rank_weights": {
            "nb8": TCN_STRUCTURED_WEIGHT,
            "tcn": 1.0 - TCN_STRUCTURED_WEIGHT,
        } if stage1_selected == "fixed_rank_blend" else None,
        "stage2_candidate": selected,
        "stage2_structured_variant": model_variant,
        "stage2_structured_features": feature_names if model_variant is not None else [],
        "stage2_structured_weight": final_weight,
        "stage2_tcn_weight": (
            1.0 - final_weight if final_weight is not None else None
        ),
        "stage2_pilot_threshold": final_threshold,
        "threshold_is_deployment_calibrated": False,
    }
    (outputs / "11_selected_schema.json").write_text(
        json.dumps(selected_schema, indent=2), encoding="utf-8"
    )

    manifest = {
        "protocol": PROTOCOL,
        "status": "complete",
        "holdout_country": HOLDOUT,
        "holdout_loaded": False,
        "selection_population": "six foreign countries only",
        "stage1_selected_branch": stage1_selected,
        "stage1_guard": stage1_guard,
        "stage1": stage1_metadata,
        "stage1_model_artifacts": copied_stage1_models,
        "stage2_selected_candidate": selected,
        "stage2_nested_outer_branch": "nested_champion",
        "stage2_available_variants": available_variants,
        "stage2_final_selection_diagnostics": final_diagnostics,
        "final_structured_variant": model_variant,
        "final_structured_weight": final_weight,
        "stage2_pilot_threshold": final_threshold,
        "threshold_is_deployment_calibrated": False,
        "guard": {
            "minimum_macro_ap_gain": 0.005,
            "minimum_improved_countries": 4,
            "maximum_worst_country_ap_drop": 0.03,
        },
        "model_artifacts": model_artifacts,
        "tcn_artifact": tcn_artifact,
        "params": PARAMS,
        "rounds": rounds,
        "tcn_epochs": tcn_epochs,
        "negative_ratio": negative_ratio,
        "pu_bags": pu_bags,
        "tcn_batch_size": tcn_batch_size,
        "pca_components": pca_components,
        "seed": seed,
        "ssl": ssl_metadata,
        "temporal": temporal_metadata,
        "nb9": nb9_metadata,
        "limitations": [
            "Stage B imagery is an enriched foreign pilot and does not estimate population precision.",
            "India imagery is unavailable, so India inference remains Stage A only.",
            "Unmatched sources are positive-unlabeled examples, not verified negatives.",
            "The Stage B pilot threshold is not a deployment threshold.",
        ],
        "elapsed_minutes": (time.time() - started) / 60,
        "python": platform.python_version(),
        "versions": {
            package: package_version(package)
            for package in [
                "numpy", "pandas", "lightgbm", "scikit-learn", "pyarrow",
                "torch", "torchgeo", "timm",
            ]
        },
    }
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return summary, metrics, predictions


if __name__ == "__main__":
    run()


## Preflight

This fails before training if the wrong notebook outputs are attached. NB2 already contains the common-window features and raw detection caches, so NB1 is not needed.


In [ ]:
from pathlib import Path
import torch

INPUT = Path("/kaggle/input")
COUNTRIES = ["Algeria", "Angola", "Indonesia", "Iraq", "Libya", "Nigeria"]

def exactly_one(name):
    matches = [path for path in INPUT.rglob(name) if path.is_file()]
    assert len(matches) == 1, f"Need exactly one {name}, found: {matches}"
    return matches[0]

assert torch.cuda.is_available(), "Set Kaggle Accelerator to GPU T4 before Save and Run All"
print("GPU:", torch.cuda.get_device_name(0))

for country in COUNTRIES:
    exactly_one(f"features_{country}_2022_2024.parquet")
    exactly_one(f"detections_{country}.parquet")

pilot = exactly_one("pilot_sources.csv")
for name in ["image_features.parquet", "image_quality.csv", "run_state.json", "feature_manifest.json"]:
    exactly_one(name)
chip_count = len(list(pilot.parent.glob("*.npz")))
assert chip_count >= 295, f"Latest NB6 v2 chips are missing, found only {chip_count}"

exactly_one("05e_manifest.json")
exactly_one("05e_loco_predictions.parquet")
stage1_models = list(INPUT.rglob("05e_final_model_*.txt"))
assert len(stage1_models) == 3, f"Need the three NB8 final models, found: {stage1_models}"

print("Input check passed")
print("NB6 chip files:", chip_count)
print("India will not be loaded")


## Run the complete foreign-country experiment

The run builds 36 monthly FIRMS bins, trains a 27,873-parameter residual TCN with two PU bags, scores the complete 1.63 million-source foreign population, and runs nested country-held-out Stage B selection. Expected free T4 runtime is mainly data aggregation and the 21 strict temporal exclusions.


In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/nb11_code")

from kg_11_multimodal import run

stage2_summary, stage2_country_metrics, stage2_predictions = run(
    input_root="/kaggle/input",
    output_root="/kaggle/working/nb11_multimodal",
    rounds=300,
    tcn_epochs=8,
    negative_ratio=10,
    pu_bags=2,
    tcn_batch_size=512,
    pca_components=16,
    seed=131,
    include_ssl=True,
)


## Results

In [ ]:
import json
import pandas as pd
from IPython.display import display

RESULT_ROOT = Path("/kaggle/working/nb11_multimodal")
OUTPUTS = RESULT_ROOT / "outputs"

stage1_country = pd.read_csv(OUTPUTS / "11_stage1_country_metrics.csv")
stage1_summary = stage1_country.groupby("branch").agg(
    macro_pr_auc=("pr_auc", "mean"),
    macro_roc_auc=("roc_auc", "mean"),
    worst_country_pr_auc=("pr_auc", "min"),
).sort_values("macro_pr_auc", ascending=False)

manifest = json.loads((OUTPUTS / "11_manifest.json").read_text())
print("Stage A selected:", manifest["stage1_selected_branch"])
print("Stage B development candidate:", manifest["stage2_selected_candidate"])
print("Stage B unbiased comparison row: nested_champion")
print("SSL status:", manifest["ssl"]["status"])

display(stage1_summary)
display(stage1_country.pivot(index="country", columns="branch", values="pr_auc"))
display(stage2_summary)
display(stage2_country_metrics.loc[
    stage2_country_metrics.branch.isin(["nb9_baseline", "nested_champion"]),
    ["country", "branch", "f1", "pr_auc", "roc_auc"],
].sort_values(["country", "branch"]))
display(pd.DataFrame(manifest["stage2_final_selection_diagnostics"]))


## Package the result

Use Save Version with Save and Run All. After the version completes, download `nb11_multimodal_results.zip` from Output.


In [ ]:
import shutil

archive = shutil.make_archive(
    "/kaggle/working/nb11_multimodal_results",
    "zip",
    root_dir="/kaggle/working/nb11_multimodal",
)
print("Created:", archive)
print("Now use Save Version, choose Save and Run All, then download the ZIP from Output.")
